# Phase 9 — Model Evaluation and Validation

## Objective
Evaluate the reliability, generalization ability, and practical limitations
of the groundwater multi-contaminant prediction models.

## Evaluation areas
1. Context-only model performance
2. Chemistry-assisted model performance
3. Spatial holdout validation
4. Residual and error distribution analysis
5. Error across target-value ranges
6. State-level and regional performance
7. Feature importance and input availability
8. Model limitations and deployment readiness

## Validation principles
- Preserve the untouched spatial test set.
- Do not train or tune models using test-set outcomes.
- Compare models using consistent evaluation records.
- Report limitations alongside performance metrics.
- Distinguish predictions from laboratory measurements.

## Source artifacts
- Phase 6 model bundle and evaluation exports
- Phase 5 target-preparation outputs
- Phase 8 testing-priority outputs

## Expected outcome
A documented model-validation report with evidence-based conclusions
about prediction reliability and deployment readiness.

**Status: IN PROGRESS**

In [1]:
# CELL 1 — Phase 9 environment setup

from pathlib import Path
import warnings
import json
import joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

warnings.filterwarnings("ignore", category=FutureWarning)

cwd = Path.cwd()

if (cwd / "data").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError(
        "Could not identify the project root. "
        "Run this notebook from the project or notebooks directory."
    )

PHASE6_DIR = PROJECT_ROOT / "reports" / "modeling" / "phase_6"
PHASE8_DIR = (
    PROJECT_ROOT / "reports"
    / "testing_prioritization" / "phase_8"
)
MODEL_DIR = PROJECT_ROOT / "models" / "phase_6"

print("PHASE 9 ENVIRONMENT")
print("Project root:", PROJECT_ROOT)
print("Phase 6 reports:", PHASE6_DIR)
print("Phase 6 models:", MODEL_DIR)
print("Phase 8 reports:", PHASE8_DIR)
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)

PHASE 9 ENVIRONMENT
Project root: c:\career\projects\BoreWell Sentinel India
Phase 6 reports: c:\career\projects\BoreWell Sentinel India\reports\modeling\phase_6
Phase 6 models: c:\career\projects\BoreWell Sentinel India\models\phase_6
Phase 8 reports: c:\career\projects\BoreWell Sentinel India\reports\testing_prioritization\phase_8
Pandas: 3.1.1
NumPy: 2.5.5


In [2]:
# CELL 2 — Artifact availability audit

EXPECTED_ARTIFACTS = {
    "phase6_model_bundle": (
        MODEL_DIR / "phase6_model_bundle.joblib"
    ),
    "phase6_model_comparison": (
        PHASE6_DIR / "phase6_final_model_comparison.csv"
    ),
    "phase6_test_predictions": (
        PHASE6_DIR / "phase6_test_predictions.csv"
    ),
    "phase6_feature_importance": (
        PHASE6_DIR / "phase6_feature_importance.csv"
    ),
    "phase6_residual_diagnostics": (
        PHASE6_DIR / "phase6_residual_diagnostics.csv"
    ),
    "phase6_error_by_value_range": (
        PHASE6_DIR / "phase6_error_by_value_range.csv"
    ),
    "phase6_chemistry_coverage": (
        PHASE6_DIR / "phase6_chemistry_assisted_coverage.csv"
    ),
    "phase8_site_priority": (
        PHASE8_DIR / "phase8_site_priority_register.csv"
    ),
    "phase8_site_actions": (
        PHASE8_DIR / "phase8_site_action_register.csv"
    ),
    "phase8_parameter_actions": (
        PHASE8_DIR / "phase8_parameter_action_register.csv"
    ),
    "phase8_sensitivity": (
        PHASE8_DIR / "phase8_weight_sensitivity_scores.csv"
    ),
    "phase8_trace_alerts": (
        PHASE8_DIR / "phase8_sparse_trace_alert_register.csv"
    ),
    "phase8_low_evidence": (
        PHASE8_DIR / "phase8_low_chemical_evidence_register.csv"
    ),
    "phase8_p1_low_evidence": (
        PHASE8_DIR / "phase8_p1_low_evidence_register.csv"
    ),
    "phase8_tier_summary": (
        PHASE8_DIR / "phase8_tier_action_summary.csv"
    ),
    "phase8_methodology": (
        PHASE8_DIR / "phase8_methodology_and_limitations.txt"
    )
}

artifact_status = []

for name, path in EXPECTED_ARTIFACTS.items():
    exists = path.exists()
    artifact_status.append({
        "Artifact": name,
        "Exists": exists,
        "Size_MB": round(path.stat().st_size / (1024 ** 2), 3)
                   if exists else None,
        "Path": str(path)
    })

artifact_audit = pd.DataFrame(artifact_status)

display(artifact_audit)

missing_phase6 = [
    name for name in EXPECTED_ARTIFACTS
    if name.startswith("phase6_")
    and not EXPECTED_ARTIFACTS[name].exists()
]

missing_phase8 = [
    name for name in EXPECTED_ARTIFACTS
    if name.startswith("phase8_")
    and not EXPECTED_ARTIFACTS[name].exists()
]

print("\nMISSING PHASE 6 ARTIFACTS:", missing_phase6)
print("MISSING PHASE 8 ARTIFACTS:", missing_phase8)

,Artifact,Exists,Size_MB,Path
0,phase6_model_bundle,True,1802.736,c:\career\projects\BoreWell Sentinel India\mod...
1,phase6_model_comparison,True,0.002,c:\career\projects\BoreWell Sentinel India\rep...
2,phase6_test_predictions,True,10.939,c:\career\projects\BoreWell Sentinel India\rep...
3,phase6_feature_importance,True,0.021,c:\career\projects\BoreWell Sentinel India\rep...
4,phase6_residual_diagnostics,True,0.001,c:\career\projects\BoreWell Sentinel India\rep...
5,phase6_error_by_value_range,True,0.003,c:\career\projects\BoreWell Sentinel India\rep...
6,phase6_chemistry_coverage,True,0.000,c:\career\projects\BoreWell Sentinel India\rep...
7,phase8_site_priority,True,15.104,c:\career\projects\BoreWell Sentinel India\rep...
8,phase8_site_actions,True,30.905,c:\career\projects\BoreWell Sentinel India\rep...
9,phase8_parameter_actions,True,66.377,c:\career\projects\BoreWell Sentinel India\rep...



MISSING PHASE 6 ARTIFACTS: []
MISSING PHASE 8 ARTIFACTS: []


In [3]:
# CELL 3 — Load available Phase 6 evaluation tables

PHASE6_TABLES = {
    "comparison": "phase6_final_model_comparison.csv",
    "predictions": "phase6_test_predictions.csv",
    "importance": "phase6_feature_importance.csv",
    "residuals": "phase6_residual_diagnostics.csv",
    "error_ranges": "phase6_error_by_value_range.csv",
    "chemistry_coverage": "phase6_chemistry_assisted_coverage.csv"
}

phase6_data = {}

for key, filename in PHASE6_TABLES.items():
    path = PHASE6_DIR / filename

    if path.exists():
        phase6_data[key] = pd.read_csv(path, low_memory=False)
        print(
            f"{key}: {phase6_data[key].shape[0]:,} rows × "
            f"{phase6_data[key].shape[1]:,} columns"
        )
    else:
        print(f"{key}: NOT AVAILABLE")

print("\nAVAILABLE TABLE SCHEMAS")

for key, df in phase6_data.items():
    print(f"\n{key.upper()}")
    print(df.columns.tolist())

if "comparison" in phase6_data:
    print("\nMODEL COMPARISON PREVIEW")
    display(phase6_data["comparison"].head(12))

if "predictions" in phase6_data:
    print("\nTEST PREDICTIONS PREVIEW")
    display(phase6_data["predictions"].head(5))

comparison: 8 rows × 10 columns
predictions: 24,708 rows × 28 columns
importance: 240 rows × 5 columns
residuals: 8 rows × 8 columns
error_ranges: 40 rows × 6 columns
chemistry_coverage: 8 rows × 5 columns

AVAILABLE TABLE SCHEMAS

COMPARISON
['Target', 'Context MAE', 'Assisted MAE', 'MAE improvement (%)', 'Context RMSE', 'Assisted RMSE', 'Context R2', 'Assisted R2', 'Context training time (seconds)', 'Assisted training time (seconds)']

PREDICTIONS
['_record_id', '_spatial_group_id', '_state_feature', '_district_feature', 'Potential of Hydrogen (pH) - Actual', 'Potential of Hydrogen (pH) - Context prediction', 'Potential of Hydrogen (pH) - Assisted prediction', 'Electric Conductivity (μS/cm) - Actual', 'Electric Conductivity (μS/cm) - Context prediction', 'Electric Conductivity (μS/cm) - Assisted prediction', 'Chloride (mg/L) - Actual', 'Chloride (mg/L) - Context prediction', 'Chloride (mg/L) - Assisted prediction', 'Bicarbonate (mg/L) - Actual', 'Bicarbonate (mg/L) - Context predicti

,Target,Context MAE,Assisted MAE,MAE improvement (%),Context RMSE,Assisted RMSE,Context R2,Assisted R2,Context training time (seconds),Assisted training time (seconds)
0,Potential of Hydrogen (pH),0.225481,0.221646,1.700993,0.322728,0.303765,0.482773,0.541770,126.895781,281.769107
1,Electric Conductivity (μS/cm),646.993948,99.282429,84.654813,1499.438654,414.032132,0.230427,0.941324,191.274661,187.085618
2,Chloride (mg/L),143.930691,32.510555,77.412354,435.924596,129.835879,0.149188,0.924525,190.154371,206.791938
3,Bicarbonate (mg/L),100.702236,51.015890,49.339863,148.878811,88.985456,0.363873,0.772744,172.182061,203.627044
4,Sulphate (mg/L),70.818798,39.824988,43.764948,239.533335,167.445332,0.120420,0.570177,178.149099,224.128448
5,Calcium (mg/L),31.748949,15.176104,52.199665,59.805880,33.053627,0.152345,0.741078,172.419345,193.807257
6,Magnesium (mg/L),24.604739,10.577967,57.008416,57.075267,34.036532,0.183536,0.709643,185.166192,191.380048
7,Sodium (mg/L),90.900383,22.062442,75.728989,250.122413,76.119140,0.227520,0.928457,185.757475,201.596999



TEST PREDICTIONS PREVIEW


,_record_id,_spatial_group_id,_state_feature,_district_feature,Potential of Hydrogen (pH) - Actual,Potential of Hydrogen (pH) - Context prediction,Potential of Hydrogen (pH) - Assisted prediction,Electric Conductivity (μS/cm) - Actual,Electric Conductivity (μS/cm) - Context prediction,Electric Conductivity (μS/cm) - Assisted prediction,...,Sulphate (mg/L) - Assisted prediction,Calcium (mg/L) - Actual,Calcium (mg/L) - Context prediction,Calcium (mg/L) - Assisted prediction,Magnesium (mg/L) - Actual,Magnesium (mg/L) - Context prediction,Magnesium (mg/L) - Assisted prediction,Sodium (mg/L) - Actual,Sodium (mg/L) - Context prediction,Sodium (mg/L) - Assisted prediction
0,gwq_chemical_parameter_manual_cgwb_an_1961_202...,andaman and nicobar islands|south andaman|anni...,andaman and nicobar islands,south andaman,7.90,7.923809,8.133822,1000.0,865.284244,1092.568459,...,45.934074,56.0,38.861165,44.363799,45.00,18.536086,28.735267,97.0,41.525007,76.364721
1,gwq_chemical_parameter_manual_cgwb_an_1961_202...,andaman and nicobar islands|south andaman|anni...,andaman and nicobar islands,south andaman,8.84,8.589350,8.262655,461.0,1017.064495,411.107692,...,15.863633,32.0,55.226960,36.255404,17.00,24.184965,18.351415,26.0,70.016415,33.396222
2,gwq_chemical_parameter_manual_cgwb_an_1961_202...,andaman and nicobar islands|south andaman|anni...,andaman and nicobar islands,south andaman,8.14,7.805765,7.980587,509.0,936.202385,513.787865,...,14.273111,44.0,56.675555,41.857716,17.00,17.918460,17.400538,31.0,72.163481,29.669831
3,gwq_chemical_parameter_manual_cgwb_an_1961_202...,andaman and nicobar islands|south andaman|anni...,andaman and nicobar islands,south andaman,8.20,8.066565,8.102925,428.0,424.168663,509.219998,...,8.492025,30.0,23.739287,21.735282,28.00,12.052379,18.977221,26.0,44.496856,21.977389
4,gwq_chemical_parameter_manual_cgwb_an_1961_202...,andaman and nicobar islands|south andaman|anni...,andaman and nicobar islands,south andaman,7.92,8.170450,8.065224,444.0,543.464878,510.672057,...,9.606155,26.0,37.182117,25.618249,25.56,12.189648,20.337728,31.0,55.360498,26.417423


In [4]:
# CELL 4 — Independent metric recalculation

prediction_df = phase6_data["predictions"].copy()
comparison_df = phase6_data["comparison"].copy()

CORE_TARGETS = [
    "Potential of Hydrogen (pH)",
    "Electric Conductivity (μS/cm)",
    "Chloride (mg/L)",
    "Bicarbonate (mg/L)",
    "Sulphate (mg/L)",
    "Calcium (mg/L)",
    "Magnesium (mg/L)",
    "Sodium (mg/L)"
]

metric_records = []
expected_records = []

for target in CORE_TARGETS:
    actual_col = f"{target} - Actual"

    actual = pd.to_numeric(
        prediction_df[actual_col], errors="coerce"
    )

    for model in ["Context", "Assisted"]:
        prediction_col = f"{target} - {model} prediction"

        predicted = pd.to_numeric(
            prediction_df[prediction_col], errors="coerce"
        )

        valid = (
            np.isfinite(actual.to_numpy())
            & np.isfinite(predicted.to_numpy())
        )

        y_true = actual.loc[valid].to_numpy()
        y_pred = predicted.loc[valid].to_numpy()

        if len(y_true) == 0:
            raise ValueError(f"No valid predictions found for {target} / {model}")

        absolute_errors = np.abs(y_true - y_pred)

        metric_records.append({
            "Target": target,
            "Model": model,
            "Records": len(y_true),
            "MAE": mean_absolute_error(y_true, y_pred),
            "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
            "R2": r2_score(y_true, y_pred),
            "Mean_signed_error": np.mean(y_pred - y_true),
            "Median_absolute_error": np.median(absolute_errors),
            "P90_absolute_error": np.percentile(absolute_errors, 90),
            "P95_absolute_error": np.percentile(absolute_errors, 95)
        })

    comparison_row = comparison_df.loc[
        comparison_df["Target"].eq(target)
    ]

    if len(comparison_row) != 1:
        raise ValueError(f"Expected one comparison row for {target}")

    comparison_row = comparison_row.iloc[0]

    for model in ["Context", "Assisted"]:
        expected_records.append({
            "Target": target,
            "Model": model,
            "Exported_MAE": comparison_row[f"{model} MAE"],
            "Exported_RMSE": comparison_row[f"{model} RMSE"],
            "Exported_R2": comparison_row[f"{model} R2"]
        })

recalculated_metrics = pd.DataFrame(metric_records)
exported_metrics = pd.DataFrame(expected_records)

metric_recheck = recalculated_metrics.merge(
    exported_metrics,
    on=["Target", "Model"],
    how="left",
    validate="one_to_one"
)

for metric in ["MAE", "RMSE", "R2"]:
    metric_recheck[f"{metric}_difference"] = (
        metric_recheck[metric] - metric_recheck[f"Exported_{metric}"]
    )

display(
    metric_recheck[
        [
            "Target", "Model", "Records",
            "MAE", "RMSE", "R2",
            "Mean_signed_error",
            "MAE_difference", "RMSE_difference", "R2_difference"
        ]
    ].round(6)
)

max_difference = metric_recheck[
    ["MAE_difference", "RMSE_difference", "R2_difference"]
].abs().max().max()

print("\nMETRIC REPRODUCIBILITY CHECK")
print("Maximum absolute difference:", max_difference)
print("Prediction rows:", len(prediction_df))
print("Unique spatial groups:", prediction_df["_spatial_group_id"].nunique())
print("States represented:", prediction_df["_state_feature"].nunique())

assert np.isfinite(max_difference)
assert max_difference < 1e-3, (
    "Recalculated metrics differ materially from the Phase 6 report. "
    "Inspect the metric table before proceeding."
)

print("PASS: Saved comparison metrics reproduced from test predictions.")

,Target,Model,Records,MAE,RMSE,R2,Mean_signed_error,MAE_difference,RMSE_difference,R2_difference
0,Potential of Hydrogen (pH),Context,24708,0.225481,0.322728,0.482773,0.004113,0.0,0.0,0.0
1,Potential of Hydrogen (pH),Assisted,24708,0.221646,0.303765,0.541770,0.000130,0.0,0.0,0.0
2,Electric Conductivity (μS/cm),Context,24708,646.993948,1499.438654,0.230427,-238.487231,0.0,0.0,0.0
3,Electric Conductivity (μS/cm),Assisted,24708,99.282429,414.032132,0.941324,-22.430001,0.0,0.0,0.0
4,Chloride (mg/L),Context,24708,143.930691,435.924596,0.149188,-73.850177,0.0,0.0,0.0
5,Chloride (mg/L),Assisted,24708,32.510555,129.835879,0.924525,-9.338598,0.0,0.0,0.0
6,Bicarbonate (mg/L),Context,24708,100.702236,148.878811,0.363873,-26.919187,0.0,0.0,0.0
7,Bicarbonate (mg/L),Assisted,24708,51.015890,88.985456,0.772744,-14.837242,0.0,0.0,0.0
8,Sulphate (mg/L),Context,24708,70.818798,239.533335,0.120420,-42.214375,0.0,0.0,0.0
9,Sulphate (mg/L),Assisted,24708,39.824988,167.445332,0.570177,-21.789486,0.0,0.0,0.0



METRIC REPRODUCIBILITY CHECK
Maximum absolute difference: 0.0
Prediction rows: 24708
Unique spatial groups: 4359
States represented: 30
PASS: Saved comparison metrics reproduced from test predictions.


In [5]:
# CELL 5 — State-level model evaluation

MIN_STATE_RECORDS = 50

state_metric_records = []

for state, state_df in prediction_df.groupby(
    "_state_feature", dropna=False
):
    for target in CORE_TARGETS:
        actual = pd.to_numeric(
            state_df[f"{target} - Actual"], errors="coerce"
        )

        for model in ["Context", "Assisted"]:
            predicted = pd.to_numeric(
                state_df[f"{target} - {model} prediction"],
                errors="coerce"
            )

            valid = (
                np.isfinite(actual.to_numpy())
                & np.isfinite(predicted.to_numpy())
            )

            y_true = actual.loc[valid].to_numpy()
            y_pred = predicted.loc[valid].to_numpy()

            if len(y_true) < MIN_STATE_RECORDS:
                continue

            state_metric_records.append({
                "State": state,
                "Target": target,
                "Model": model,
                "Records": len(y_true),
                "MAE": mean_absolute_error(y_true, y_pred),
                "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
                "R2": r2_score(y_true, y_pred),
                "Mean_signed_error": np.mean(y_pred - y_true)
            })

state_metrics = pd.DataFrame(state_metric_records)

if state_metrics.empty:
    raise ValueError(
        "No state-target combinations met the minimum record threshold."
    )

regional_summary = (
    state_metrics
    .groupby(["Target", "Model"])
    .agg(
        States_evaluated=("State", "nunique"),
        Test_records=("Records", "sum"),
        Median_state_MAE=("MAE", "median"),
        Median_state_RMSE=("RMSE", "median"),
        Median_state_R2=("R2", "median"),
        Minimum_state_R2=("R2", "min"),
        Maximum_state_R2=("R2", "max")
    )
    .reset_index()
)

print("STATE-LEVEL EVALUATION SUMMARY")
display(regional_summary.round(4))

print("\nLOWEST-R2 STATE RESULTS")
weakest_state_results = (
    state_metrics
    .sort_values("R2", ascending=True)
    .groupby(["Target", "Model"], group_keys=False)
    .head(3)
)

display(weakest_state_results.round(4))

print("\nREGIONAL COVERAGE")
print("States with at least one evaluated target:",
      state_metrics["State"].nunique())
print("State-target-model evaluations:", len(state_metrics))
print("Minimum records per state-target:", MIN_STATE_RECORDS)

STATE-LEVEL EVALUATION SUMMARY


,Target,Model,States_evaluated,Test_records,Median_state_MAE,Median_state_RMSE,Median_state_R2,Minimum_state_R2,Maximum_state_R2
0,Bicarbonate (mg/L),Assisted,24,24623,44.2477,73.2057,0.7198,0.5233,0.8616
1,Bicarbonate (mg/L),Context,24,24623,88.1736,128.9195,0.2247,-0.0129,0.4296
2,Calcium (mg/L),Assisted,24,24623,12.2748,22.9779,0.7324,0.3734,0.8710
3,Calcium (mg/L),Context,24,24623,23.5190,41.8660,0.1046,-0.1228,0.3600
4,Chloride (mg/L),Assisted,24,24623,21.4197,43.0221,0.9386,0.7042,0.9719
5,Chloride (mg/L),Context,24,24623,72.7510,181.4851,0.0746,-0.1837,0.3113
6,Electric Conductivity (μS/cm),Assisted,24,24623,83.9027,191.7126,0.9491,0.7230,0.9911
7,Electric Conductivity (μS/cm),Context,24,24623,359.2102,694.8114,0.1560,-0.2190,0.5450
8,Magnesium (mg/L),Assisted,24,24623,7.4919,14.3910,0.7941,0.3992,0.9122
9,Magnesium (mg/L),Context,24,24623,16.1865,30.3613,0.1349,-0.0775,0.6674



LOWEST-R2 STATE RESULTS


,State,Target,Model,Records,MAE,RMSE,R2,Mean_signed_error
2,andaman and nicobar islands,Electric Conductivity (μS/cm),Context,112,232.8286,286.2027,-0.2190,19.5598
4,andaman and nicobar islands,Chloride (mg/L),Context,112,27.0092,39.8091,-0.1837,-3.6739
14,andaman and nicobar islands,Sodium (mg/L),Context,112,25.9495,37.9728,-0.1486,-5.1152
90,delhi,Calcium (mg/L),Context,199,77.9478,148.3670,-0.1228,-53.6766
180,jharkhand,Chloride (mg/L),Context,185,48.0741,73.5393,-0.1126,-12.9433
88,delhi,Sulphate (mg/L),Context,199,293.9446,580.3615,-0.1015,-228.2629
52,bihar,Chloride (mg/L),Context,161,56.2653,113.4434,-0.0785,-32.4056
12,andaman and nicobar islands,Magnesium (mg/L),Context,112,13.5908,20.7138,-0.0775,-8.0371
92,delhi,Magnesium (mg/L),Context,199,83.8077,158.5700,-0.0529,-62.8581
298,rajasthan,Calcium (mg/L),Context,2081,39.6589,68.0795,-0.0370,-16.9762



REGIONAL COVERAGE
States with at least one evaluated target: 24
State-target-model evaluations: 384
Minimum records per state-target: 50


In [6]:
# CELL 6 — Residual and value-range diagnostic review

print("OVERALL RESIDUAL DIAGNOSTICS")
display(
    phase6_data["residuals"]
    .sort_values("Target")
    .round(4)
)

print("\nERROR BY ACTUAL-VALUE RANGE")
display(
    phase6_data["error_ranges"]
    .sort_values(["Target", "Actual-value band"])
    .round(4)
)

print("\nASSISTED MODEL ERROR SUMMARY")
assisted_error_summary = (
    recalculated_metrics.loc[
        recalculated_metrics["Model"].eq("Assisted"),
        [
            "Target", "MAE", "RMSE",
            "Median_absolute_error",
            "P90_absolute_error", "P95_absolute_error",
            "Mean_signed_error"
        ]
    ]
    .sort_values("P95_absolute_error", ascending=False)
)

display(assisted_error_summary.round(4))

OVERALL RESIDUAL DIAGNOSTICS


,Target,Mean signed error,Median absolute error,90th percentile absolute error,95th percentile absolute error,Maximum absolute error,Mean actual,Mean predicted
3,Bicarbonate (mg/L),-26.9192,68.9137,223.9894,298.2440,1832.5488,289.0518,262.1327
5,Calcium (mg/L),-11.9112,17.4199,67.5982,106.2767,954.7186,65.2394,53.3282
2,Chloride (mg/L),-73.8502,44.6800,317.9503,559.2517,17642.3987,204.1338,130.2836
1,Electric Conductivity (μS/cm),-238.4872,282.8351,1463.9055,2373.4498,43223.3841,1310.8812,1072.3939
6,Magnesium (mg/L),-10.7951,11.7902,53.4091,86.7421,2154.8104,44.7746,33.9795
0,Potential of Hydrogen (pH),0.0041,0.1675,0.4788,0.6113,5.2019,7.8170,7.8211
7,Sodium (mg/L),-42.6675,28.4190,212.3996,362.3820,9293.1903,142.4217,99.7542
4,Sulphate (mg/L),-42.2144,20.2080,152.6491,281.3592,17168.9492,97.8616,55.6472



ERROR BY ACTUAL-VALUE RANGE


,Target,Actual-value band,Records,Actual_median,Predicted_median,MAE
15,Bicarbonate (mg/L),"(-0.001, 134.216]",4942,78.0000,124.2672,71.4182
16,Bicarbonate (mg/L),"(134.216, 220.0]",4949,183.0000,214.5883,70.7727
17,Bicarbonate (mg/L),"(220.0, 305.0]",5148,262.3000,269.6585,68.2776
18,Bicarbonate (mg/L),"(305.0, 415.0]",4840,359.4500,308.2346,82.6768
19,Bicarbonate (mg/L),"(415.0, 2313.577]",4829,522.0000,356.9994,213.9779
25,Calcium (mg/L),"(-0.001, 24.0]",5019,16.0000,27.9414,17.4646
26,Calcium (mg/L),"(24.0, 40.0]",5191,32.7053,42.0238,16.8009
27,Calcium (mg/L),"(40.0, 59.278]",4620,49.0000,51.2344,16.5149
28,Calcium (mg/L),"(59.278, 89.0]",4942,72.0000,58.5891,20.9934
29,Calcium (mg/L),"(89.0, 1160.0]",4936,128.0000,68.1546,87.0212



ASSISTED MODEL ERROR SUMMARY


,Target,MAE,RMSE,Median_absolute_error,P90_absolute_error,P95_absolute_error,Mean_signed_error
3,Electric Conductivity (μS/cm),99.2824,414.0321,40.9684,205.3725,334.3281,-22.4300
7,Bicarbonate (mg/L),51.0159,88.9855,28.5602,118.2564,170.6462,-14.8372
9,Sulphate (mg/L),39.8250,167.4453,13.2155,86.5670,148.3972,-21.7895
5,Chloride (mg/L),32.5106,129.8359,12.4652,70.9747,112.5216,-9.3386
15,Sodium (mg/L),22.0624,76.1191,8.4440,47.7938,79.4064,-7.9226
11,Calcium (mg/L),15.1761,33.0536,7.0052,33.9856,53.5227,-5.0667
13,Magnesium (mg/L),10.5780,34.0365,4.4211,23.1130,36.8236,-4.0186
1,Potential of Hydrogen (pH),0.2216,0.3038,0.1694,0.4699,0.5966,0.0001


In [7]:
# CELL 7 — Residual bias across actual-value deciles

decile_records = []

for target in CORE_TARGETS:
    actual = pd.to_numeric(
        prediction_df[f"{target} - Actual"], errors="coerce"
    )

    for model in ["Context", "Assisted"]:
        predicted = pd.to_numeric(
            prediction_df[f"{target} - {model} prediction"],
            errors="coerce"
        )

        valid = actual.notna() & predicted.notna()

        diagnostic = pd.DataFrame({
            "Actual": actual.loc[valid],
            "Predicted": predicted.loc[valid]
        })

        diagnostic["Signed_error"] = (
            diagnostic["Predicted"] - diagnostic["Actual"]
        )
        diagnostic["Absolute_error"] = (
            diagnostic["Signed_error"].abs()
        )

        # Rank first so repeated measurements do not cause
        # qcut to collapse all decile boundaries.
        diagnostic["Actual_decile"] = pd.qcut(
            diagnostic["Actual"].rank(method="first"),
            q=10,
            labels=False
        ) + 1

        grouped = diagnostic.groupby("Actual_decile")

        for decile, group in grouped:
            decile_records.append({
                "Target": target,
                "Model": model,
                "Actual_decile": int(decile),
                "Records": len(group),
                "Actual_median": group["Actual"].median(),
                "Predicted_median": group["Predicted"].median(),
                "Mean_signed_error": group["Signed_error"].mean(),
                "MAE": group["Absolute_error"].mean(),
                "Underprediction_percentage": (
                    group["Signed_error"].lt(0).mean() * 100
                )
            })

decile_diagnostics = pd.DataFrame(decile_records)

print("ASSISTED MODEL — HIGHEST ACTUAL-VALUE DECILE")
display(
    decile_diagnostics.loc[
        (decile_diagnostics["Model"] == "Assisted")
        & (decile_diagnostics["Actual_decile"] == 10)
    ]
    .sort_values("Target")
    .round(3)
)

print("\nASSISTED MODEL — ALL DECILES")
display(
    decile_diagnostics.loc[
        decile_diagnostics["Model"].eq("Assisted")
    ]
    .sort_values(["Target", "Actual_decile"])
    .round(3)
)

ASSISTED MODEL — HIGHEST ACTUAL-VALUE DECILE


,Target,Model,Actual_decile,Records,Actual_median,Predicted_median,Mean_signed_error,MAE,Underprediction_percentage
79,Bicarbonate (mg/L),Assisted,10,2471,622.000,528.756,-136.245,147.431,89.680
119,Calcium (mg/L),Assisted,10,2471,173.747,141.597,-48.618,57.983,87.657
59,Chloride (mg/L),Assisted,10,2471,780.000,745.575,-77.921,143.867,73.412
39,Electric Conductivity (μS/cm),Assisted,10,2471,3930.000,3826.011,-187.236,405.028,58.236
139,Magnesium (mg/L),Assisted,10,2471,134.000,113.620,-36.883,46.683,83.691
19,Potential of Hydrogen (pH),Assisted,10,2471,8.500,8.173,-0.381,0.391,94.537
159,Sodium (mg/L),Assisted,10,2471,542.000,515.080,-58.049,94.094,75.637
99,Sulphate (mg/L),Assisted,10,2471,389.000,291.788,-175.706,207.224,86.564



ASSISTED MODEL — ALL DECILES


,Target,Model,Actual_decile,Records,Actual_median,Predicted_median,Mean_signed_error,MAE,Underprediction_percentage
70,Bicarbonate (mg/L),Assisted,1,2471,42.0,45.522,18.348,25.044,38.486
71,Bicarbonate (mg/L),Assisted,2,2471,110.0,118.482,23.200,37.551,43.181
72,Bicarbonate (mg/L),Assisted,3,2471,159.0,163.146,13.028,35.033,49.211
73,Bicarbonate (mg/L),Assisted,4,2470,201.0,201.906,9.548,36.078,51.903
74,Bicarbonate (mg/L),Assisted,5,2471,244.0,238.171,4.665,37.564,55.200
...,...,...,...,...,...,...,...,...,...
95,Sulphate (mg/L),Assisted,6,2471,43.0,33.230,-2.716,19.328,68.960
96,Sulphate (mg/L),Assisted,7,2470,60.0,45.351,-8.087,25.815,73.401
97,Sulphate (mg/L),Assisted,8,2471,90.0,64.000,-17.584,36.012,78.430
98,Sulphate (mg/L),Assisted,9,2471,150.0,105.652,-36.264,59.267,80.453


In [8]:
# CELL 8 — State-level head-to-head comparison

state_comparison = state_metrics.pivot(
    index=["State", "Target"],
    columns="Model",
    values=["MAE", "R2"]
)

state_comparison.columns = [
    f"{metric}_{model}"
    for metric, model in state_comparison.columns
]

state_comparison = state_comparison.reset_index()

state_comparison["Assisted_MAE_better"] = (
    state_comparison["MAE_Assisted"]
    < state_comparison["MAE_Context"]
)

state_comparison["Assisted_R2_better"] = (
    state_comparison["R2_Assisted"]
    > state_comparison["R2_Context"]
)

state_win_summary = (
    state_comparison
    .groupby("Target")
    .agg(
        States_compared=("State", "nunique"),
        Assisted_MAE_wins=("Assisted_MAE_better", "sum"),
        Assisted_R2_wins=("Assisted_R2_better", "sum")
    )
    .reset_index()
)

state_win_summary["MAE_win_percentage"] = (
    state_win_summary["Assisted_MAE_wins"]
    / state_win_summary["States_compared"] * 100
)

state_win_summary["R2_win_percentage"] = (
    state_win_summary["Assisted_R2_wins"]
    / state_win_summary["States_compared"] * 100
)

print("STATE-LEVEL MODEL WIN SUMMARY")
display(state_win_summary.round(2))

print("\nSTATE COVERAGE AUDIT")

state_record_counts = (
    prediction_df.groupby("_state_feature", dropna=False)
    .size()
    .sort_values(ascending=False)
    .rename("Test_records")
    .reset_index()
)

print("Total test records:", len(prediction_df))
print("Records in states meeting minimum threshold:",
      int(state_metrics.drop_duplicates(
          ["State", "Target", "Model"]
      ).groupby("State")["Records"].first().sum())
      if False else
      int(state_comparison["State"].nunique()))

print("States represented in test predictions:",
      prediction_df["_state_feature"].nunique())

display(state_record_counts)

STATE-LEVEL MODEL WIN SUMMARY


,Target,States_compared,Assisted_MAE_wins,Assisted_R2_wins,MAE_win_percentage,R2_win_percentage
0,Bicarbonate (mg/L),24,24,24,100.00,100.00
1,Calcium (mg/L),24,24,24,100.00,100.00
2,Chloride (mg/L),24,24,24,100.00,100.00
3,Electric Conductivity (μS/cm),24,24,24,100.00,100.00
4,Magnesium (mg/L),24,24,24,100.00,100.00
5,Potential of Hydrogen (pH),24,9,10,37.50,41.67
6,Sodium (mg/L),24,24,24,100.00,100.00
7,Sulphate (mg/L),24,23,23,95.83,95.83



STATE COVERAGE AUDIT
Total test records: 24708
Records in states meeting minimum threshold: 24
States represented in test predictions: 30


,_state_feature,Test_records
0,madhya pradesh,3411
1,maharashtra,3059
2,rajasthan,2081
3,chhattisgarh,1812
4,gujarat,1732
5,odisha,1435
6,andhra pradesh,1405
7,tamil nadu,1366
8,uttar pradesh,1214
9,haryana,1120


In [9]:
# CELL 9 — Correct state coverage and exclusion audit

state_coverage_audit = (
    state_record_counts
    .assign(
        Meets_minimum=lambda x: (
            x["Test_records"] >= MIN_STATE_RECORDS
        )
    )
    .sort_values("Test_records", ascending=False)
)

eligible_states = state_coverage_audit.loc[
    state_coverage_audit["Meets_minimum"]
]

excluded_states = state_coverage_audit.loc[
    ~state_coverage_audit["Meets_minimum"]
]

eligible_records = int(eligible_states["Test_records"].sum())
excluded_records = int(excluded_states["Test_records"].sum())

print("STATE COVERAGE AUDIT")
print("States represented:", len(state_coverage_audit))
print("States meeting minimum:", len(eligible_states))
print("States excluded:", len(excluded_states))
print("Total test records:", len(prediction_df))
print("Records in eligible states:", eligible_records)
print("Records in excluded states:", excluded_records)

print("\nEXCLUDED SMALL STATE GROUPS")
display(excluded_states)

assert eligible_records + excluded_records == len(prediction_df)
assert len(eligible_states) == 24
assert excluded_records == 85

print("\nPASS: State coverage reconciled.")

STATE COVERAGE AUDIT
States represented: 30
States meeting minimum: 24
States excluded: 6
Total test records: 24708
Records in eligible states: 24623
Records in excluded states: 85

EXCLUDED SMALL STATE GROUPS


,_state_feature,Test_records,Meets_minimum
24,tripura,40,False
25,meghalaya,29,False
26,puducherry,9,False
27,arunachal pradesh,3,False
28,dadra and nagar haveli,3,False
29,nagaland,1,False



PASS: State coverage reconciled.


In [10]:
# CELL 10 — Low vs high actual-value decile diagnostics

tail_records = []

for target in CORE_TARGETS:
    for model in ["Context", "Assisted"]:

        selected = decile_diagnostics.loc[
            (decile_diagnostics["Target"] == target)
            & (decile_diagnostics["Model"] == model)
        ]

        overall = recalculated_metrics.loc[
            (recalculated_metrics["Target"] == target)
            & (recalculated_metrics["Model"] == model)
        ].iloc[0]

        for decile, label in [(1, "Lowest"), (10, "Highest")]:
            row = selected.loc[
                selected["Actual_decile"] == decile
            ].iloc[0]

            tail_records.append({
                "Target": target,
                "Model": model,
                "Value_range": label,
                "Records": row["Records"],
                "Actual_median": row["Actual_median"],
                "Predicted_median": row["Predicted_median"],
                "Mean_signed_error": row["Mean_signed_error"],
                "MAE": row["MAE"],
                "Overall_MAE": overall["MAE"],
                "Tail_to_overall_MAE_ratio": (
                    row["MAE"] / overall["MAE"]
                ),
                "Underprediction_percentage": (
                    row["Underprediction_percentage"]
                )
            })

tail_comparison = pd.DataFrame(tail_records)

print("HIGHEST-VALUE DECILE — BOTH MODELS")
display(
    tail_comparison.loc[
        tail_comparison["Value_range"] == "Highest"
    ]
    .sort_values(["Target", "Model"])
    .round(3)
)

print("\nLOWEST-VALUE DECILE — BOTH MODELS")
display(
    tail_comparison.loc[
        tail_comparison["Value_range"] == "Lowest"
    ]
    .sort_values(["Target", "Model"])
    .round(3)
)

print("\nHIGHEST-DECILE ERROR RELATIVE TO OVERALL ERROR")
display(
    tail_comparison.loc[
        tail_comparison["Value_range"] == "Highest",
        [
            "Target", "Model", "MAE",
            "Overall_MAE", "Tail_to_overall_MAE_ratio",
            "Underprediction_percentage"
        ]
    ]
    .sort_values(["Target", "Model"])
    .round(3)
)

HIGHEST-VALUE DECILE — BOTH MODELS


,Target,Model,Value_range,Records,Actual_median,Predicted_median,Mean_signed_error,MAE,Overall_MAE,Tail_to_overall_MAE_ratio,Underprediction_percentage
15,Bicarbonate (mg/L),Assisted,Highest,2471,622.000,528.756,-136.245,147.431,51.016,2.890,89.680
13,Bicarbonate (mg/L),Context,Highest,2471,622.000,379.348,-291.267,293.592,100.702,2.915,97.289
23,Calcium (mg/L),Assisted,Highest,2471,173.747,141.597,-48.618,57.983,15.176,3.821,87.657
21,Calcium (mg/L),Context,Highest,2471,173.747,71.314,-129.352,133.126,31.749,4.193,98.219
11,Chloride (mg/L),Assisted,Highest,2471,780.000,745.575,-77.921,143.867,32.511,4.425,73.412
9,Chloride (mg/L),Context,Highest,2471,780.000,257.515,-766.454,809.587,143.931,5.625,94.253
7,Electric Conductivity (μS/cm),Assisted,Highest,2471,3930.000,3826.011,-187.236,405.028,99.282,4.080,58.236
5,Electric Conductivity (μS/cm),Context,Highest,2471,3930.000,1854.696,-2818.957,2978.576,646.994,4.604,93.889
27,Magnesium (mg/L),Assisted,Highest,2471,134.000,113.620,-36.883,46.683,10.578,4.413,83.691
25,Magnesium (mg/L),Context,Highest,2471,134.000,52.884,-113.867,115.223,24.605,4.683,97.815



LOWEST-VALUE DECILE — BOTH MODELS


,Target,Model,Value_range,Records,Actual_median,Predicted_median,Mean_signed_error,MAE,Overall_MAE,Tail_to_overall_MAE_ratio,Underprediction_percentage
14,Bicarbonate (mg/L),Assisted,Lowest,2471,42.00,45.522,18.348,25.044,51.016,0.491,38.486
12,Bicarbonate (mg/L),Context,Lowest,2471,42.00,68.353,53.795,59.621,100.702,0.592,19.911
22,Calcium (mg/L),Assisted,Lowest,2471,10.00,13.744,6.055,6.800,15.176,0.448,22.380
20,Calcium (mg/L),Context,Lowest,2471,10.00,21.569,16.155,16.782,31.749,0.529,10.239
10,Chloride (mg/L),Assisted,Lowest,2471,11.00,14.671,6.440,6.927,32.511,0.213,17.321
8,Chloride (mg/L),Context,Lowest,2471,11.00,27.405,30.001,30.189,143.931,0.210,3.764
6,Electric Conductivity (μS/cm),Assisted,Lowest,2471,172.00,178.201,15.983,29.350,99.282,0.296,43.302
4,Electric Conductivity (μS/cm),Context,Lowest,2471,172.00,264.906,177.342,193.766,646.994,0.299,16.512
26,Magnesium (mg/L),Assisted,Lowest,2471,3.60,4.615,2.740,3.235,10.578,0.306,27.681
24,Magnesium (mg/L),Context,Lowest,2471,3.60,8.191,8.846,9.182,24.605,0.373,13.679



HIGHEST-DECILE ERROR RELATIVE TO OVERALL ERROR


,Target,Model,MAE,Overall_MAE,Tail_to_overall_MAE_ratio,Underprediction_percentage
15,Bicarbonate (mg/L),Assisted,147.431,51.016,2.890,89.680
13,Bicarbonate (mg/L),Context,293.592,100.702,2.915,97.289
23,Calcium (mg/L),Assisted,57.983,15.176,3.821,87.657
21,Calcium (mg/L),Context,133.126,31.749,4.193,98.219
11,Chloride (mg/L),Assisted,143.867,32.511,4.425,73.412
9,Chloride (mg/L),Context,809.587,143.931,5.625,94.253
7,Electric Conductivity (μS/cm),Assisted,405.028,99.282,4.080,58.236
5,Electric Conductivity (μS/cm),Context,2978.576,646.994,4.604,93.889
27,Magnesium (mg/L),Assisted,46.683,10.578,4.413,83.691
25,Magnesium (mg/L),Context,115.223,24.605,4.683,97.815


In [11]:
# CELL 11 — Load Phase 5 screening threshold references

PHASE5_DIR = PROJECT_ROOT / "reports" / "feature_engineering"

THRESHOLD_PATH = PHASE5_DIR / "threshold_reference_register.csv"
LABEL_SUPPORT_PATH = PHASE5_DIR / "classification_label_support.csv"
CLASSIFICATION_PATH = PHASE5_DIR / "classification_context_labels.csv"

for path in [
    THRESHOLD_PATH,
    LABEL_SUPPORT_PATH,
    CLASSIFICATION_PATH
]:
    if not path.exists():
        raise FileNotFoundError(f"Required Phase 5 artifact missing: {path}")

threshold_registry = pd.read_csv(THRESHOLD_PATH)
classification_support = pd.read_csv(LABEL_SUPPORT_PATH)

# Read only a small preview to inspect classification column names.
classification_preview = pd.read_csv(
    CLASSIFICATION_PATH,
    nrows=5
)

print("THRESHOLD REGISTRY")
print("Shape:", threshold_registry.shape)
print("Columns:", threshold_registry.columns.tolist())
display(threshold_registry)

print("\nCLASSIFICATION LABEL SUPPORT")
print("Shape:", classification_support.shape)
print("Columns:", classification_support.columns.tolist())
display(classification_support)

print("\nCLASSIFICATION CONTEXT SCHEMA")
print("Preview shape:", classification_preview.shape)
print(classification_preview.columns.tolist())

THRESHOLD REGISTRY
Shape: (19, 8)
Columns: ['Parameter', 'Unit', 'Acceptable_Lower', 'Acceptable_Upper', 'Acceptable_Max', 'Permissible_Max', 'Rule', 'Status']


,Parameter,Unit,Acceptable_Lower,Acceptable_Upper,Acceptable_Max,Permissible_Max,Rule,Status
0,Potential of Hydrogen (pH),pH,6.5,8.5,NaN,NaN,Range; no relaxation,Ready
1,Total Dissolved Solids (mg/L),mg/L,NaN,NaN,500.000,2000.0,Upper limit,Ready
2,Chloride (mg/L),mg/L,NaN,NaN,250.000,1000.0,Upper limit,Ready
3,Sulphate (mg/L),mg/L,NaN,NaN,200.000,400.0,Conditional on Mg <= 30 mg/L,Ready with condition
4,Total Hardness (mgCaCO3/L),mg/L as CaCO3,NaN,NaN,200.000,600.0,Upper limit,Ready
5,Calcium (mg/L),mg/L as Ca,NaN,NaN,75.000,200.0,Upper limit,Ready
6,Magnesium (mg/L),mg/L as Mg,NaN,NaN,30.000,100.0,Upper limit,Ready
7,Total Alkalinity (mg/L as CaCO3),mg/L as CaCO3,NaN,NaN,200.000,600.0,Upper limit,Ready
8,Arsenic (mg/L),mg/L as As,NaN,NaN,0.010,NaN,No relaxation,Provisional analyte match
9,Cadmium (mg/L),mg/L as Cd,NaN,NaN,0.003,NaN,No relaxation,Ready



CLASSIFICATION LABEL SUPPORT
Shape: (31, 5)
Columns: ['Label', 'Available_Labels', 'Positive_Count', 'Negative_Count', 'Positive_Class_Percentage']


,Label,Available_Labels,Positive_Count,Negative_Count,Positive_Class_Percentage
0,Potential of Hydrogen (pH)__exceeds_acceptable,162719,10399,152320,6.391
1,Potential of Hydrogen (pH)__exceeds_permissible,162719,10399,152320,6.391
2,Chloride (mg/L)__exceeds_acceptable,162033,29757,132276,18.365
3,Chloride (mg/L)__exceeds_permissible,162033,5008,157025,3.091
4,Magnesium (mg/L)__exceeds_acceptable,150412,70219,80193,46.684
5,Magnesium (mg/L)__exceeds_permissible,150412,11423,138989,7.594
6,Calcium (mg/L)__exceeds_acceptable,149855,38451,111404,25.659
7,Calcium (mg/L)__exceeds_permissible,149855,4622,145233,3.084
8,Total Hardness (mgCaCO3/L)__exceeds_acceptable,144408,88597,55811,61.352
9,Total Hardness (mgCaCO3/L)__exceeds_permissible,144408,14789,129619,10.241



CLASSIFICATION CONTEXT SCHEMA
Preview shape: (5, 50)
['_record_id', '_source_file', '_source_row', '_parsed_date', '_provisional_station_id', '_provisional_location_candidate_id', '_spatial_group_id', '_geo_grid_id', '_latitude', '_longitude', '_coordinates_missing', '_state_feature', '_district_feature', '_state_district_feature', '_year_index', '_month_sin', '_month_cos', '_dayofyear_sin', '_dayofyear_cos', 'Potential of Hydrogen (pH)__exceeds_acceptable', 'Potential of Hydrogen (pH)__exceeds_permissible', 'Chloride (mg/L)__exceeds_acceptable', 'Chloride (mg/L)__exceeds_permissible', 'Magnesium (mg/L)__exceeds_acceptable', 'Magnesium (mg/L)__exceeds_permissible', 'Calcium (mg/L)__exceeds_acceptable', 'Calcium (mg/L)__exceeds_permissible', 'Total Hardness (mgCaCO3/L)__exceeds_acceptable', 'Total Hardness (mgCaCO3/L)__exceeds_permissible', 'Sulphate (mg/L)__exceeds_acceptable', 'Sulphate (mg/L)__exceeds_permissible', 'Total Alkalinity (mg/L as CaCO3)__exceeds_acceptable', 'Total Alkal

In [12]:
# CELL 12 — Identify the exact screening definitions for core parameters

print("THRESHOLD DEFINITIONS")
display(threshold_registry)

print("\nCLASSIFICATION LABELS")
label_columns = [
    col for col in classification_preview.columns
    if col not in [
        "_record_id",
        "_spatial_group_id",
        "_state_feature",
        "_district_feature",
        "_latitude",
        "_longitude",
        "_coordinates_missing"
    ]
]

for col in label_columns:
    print("-", col)

print("\nLABEL SUPPORT DETAILS")
display(classification_support)

THRESHOLD DEFINITIONS


,Parameter,Unit,Acceptable_Lower,Acceptable_Upper,Acceptable_Max,Permissible_Max,Rule,Status
0,Potential of Hydrogen (pH),pH,6.5,8.5,NaN,NaN,Range; no relaxation,Ready
1,Total Dissolved Solids (mg/L),mg/L,NaN,NaN,500.000,2000.0,Upper limit,Ready
2,Chloride (mg/L),mg/L,NaN,NaN,250.000,1000.0,Upper limit,Ready
3,Sulphate (mg/L),mg/L,NaN,NaN,200.000,400.0,Conditional on Mg <= 30 mg/L,Ready with condition
4,Total Hardness (mgCaCO3/L),mg/L as CaCO3,NaN,NaN,200.000,600.0,Upper limit,Ready
5,Calcium (mg/L),mg/L as Ca,NaN,NaN,75.000,200.0,Upper limit,Ready
6,Magnesium (mg/L),mg/L as Mg,NaN,NaN,30.000,100.0,Upper limit,Ready
7,Total Alkalinity (mg/L as CaCO3),mg/L as CaCO3,NaN,NaN,200.000,600.0,Upper limit,Ready
8,Arsenic (mg/L),mg/L as As,NaN,NaN,0.010,NaN,No relaxation,Provisional analyte match
9,Cadmium (mg/L),mg/L as Cd,NaN,NaN,0.003,NaN,No relaxation,Ready



CLASSIFICATION LABELS
- _source_file
- _source_row
- _parsed_date
- _provisional_station_id
- _provisional_location_candidate_id
- _geo_grid_id
- _state_district_feature
- _year_index
- _month_sin
- _month_cos
- _dayofyear_sin
- _dayofyear_cos
- Potential of Hydrogen (pH)__exceeds_acceptable
- Potential of Hydrogen (pH)__exceeds_permissible
- Chloride (mg/L)__exceeds_acceptable
- Chloride (mg/L)__exceeds_permissible
- Magnesium (mg/L)__exceeds_acceptable
- Magnesium (mg/L)__exceeds_permissible
- Calcium (mg/L)__exceeds_acceptable
- Calcium (mg/L)__exceeds_permissible
- Total Hardness (mgCaCO3/L)__exceeds_acceptable
- Total Hardness (mgCaCO3/L)__exceeds_permissible
- Sulphate (mg/L)__exceeds_acceptable
- Sulphate (mg/L)__exceeds_permissible
- Total Alkalinity (mg/L as CaCO3)__exceeds_acceptable
- Total Alkalinity (mg/L as CaCO3)__exceeds_permissible
- Total Dissolved Solids (mg/L)__exceeds_acceptable
- Total Dissolved Solids (mg/L)__exceeds_permissible
- Arsenic (mg/L)__exceeds_accepta

,Label,Available_Labels,Positive_Count,Negative_Count,Positive_Class_Percentage
0,Potential of Hydrogen (pH)__exceeds_acceptable,162719,10399,152320,6.391
1,Potential of Hydrogen (pH)__exceeds_permissible,162719,10399,152320,6.391
2,Chloride (mg/L)__exceeds_acceptable,162033,29757,132276,18.365
3,Chloride (mg/L)__exceeds_permissible,162033,5008,157025,3.091
4,Magnesium (mg/L)__exceeds_acceptable,150412,70219,80193,46.684
5,Magnesium (mg/L)__exceeds_permissible,150412,11423,138989,7.594
6,Calcium (mg/L)__exceeds_acceptable,149855,38451,111404,25.659
7,Calcium (mg/L)__exceeds_permissible,149855,4622,145233,3.084
8,Total Hardness (mgCaCO3/L)__exceeds_acceptable,144408,88597,55811,61.352
9,Total Hardness (mgCaCO3/L)__exceeds_permissible,144408,14789,129619,10.241


In [13]:
# CELL 13 — Load threshold labels for the untouched test set

SCREENING_TARGETS = {
    "Potential of Hydrogen (pH)": {
        "prediction_prefix": "Potential of Hydrogen (pH)",
        "labels": ["exceeds_acceptable"]
    },
    "Chloride (mg/L)": {
        "prediction_prefix": "Chloride (mg/L)",
        "labels": ["exceeds_acceptable", "exceeds_permissible"]
    },
    "Calcium (mg/L)": {
        "prediction_prefix": "Calcium (mg/L)",
        "labels": ["exceeds_acceptable", "exceeds_permissible"]
    },
    "Magnesium (mg/L)": {
        "prediction_prefix": "Magnesium (mg/L)",
        "labels": ["exceeds_acceptable", "exceeds_permissible"]
    },
    "Sulphate (mg/L)": {
        "prediction_prefix": "Sulphate (mg/L)",
        "labels": ["exceeds_acceptable", "exceeds_permissible"]
    }
}

SCREENING_LABEL_COLUMNS = ["_record_id"]

for parameter, spec in SCREENING_TARGETS.items():
    for suffix in spec["labels"]:
        SCREENING_LABEL_COLUMNS.append(
            f"{parameter}__{suffix}"
        )

classification_flags = pd.read_csv(
    CLASSIFICATION_PATH,
    usecols=SCREENING_LABEL_COLUMNS,
    low_memory=False
)

classification_flags["_record_id"] = (
    classification_flags["_record_id"].astype(str)
)

screening_test_df = prediction_df.copy()
screening_test_df["_record_id"] = (
    screening_test_df["_record_id"].astype(str)
)

screening_test_df = screening_test_df.merge(
    classification_flags,
    on="_record_id",
    how="left",
    validate="one_to_one",
    indicator=True
)

print("SCREENING TEST JOIN")
print("Prediction rows:", len(prediction_df))
print("Joined rows:", len(screening_test_df))
print("Matched:", int(screening_test_df["_merge"].eq("both").sum()))
print("Unmatched:", int(screening_test_df["_merge"].eq("left_only").sum()))

assert len(screening_test_df) == len(prediction_df)
assert screening_test_df["_merge"].eq("both").all()

screening_test_df.drop(columns="_merge", inplace=True)

print("\nSCREENING LABEL AVAILABILITY")
for col in SCREENING_LABEL_COLUMNS[1:]:
    print(
        col,
        "| available:", int(screening_test_df[col].notna().sum()),
        "| positive:", int(screening_test_df[col].eq(1).sum())
    )

SCREENING TEST JOIN
Prediction rows: 24708
Joined rows: 24708
Matched: 24708
Unmatched: 0

SCREENING LABEL AVAILABILITY
Potential of Hydrogen (pH)__exceeds_acceptable | available: 24708 | positive: 1292
Chloride (mg/L)__exceeds_acceptable | available: 24708 | positive: 4758
Chloride (mg/L)__exceeds_permissible | available: 24708 | positive: 867
Calcium (mg/L)__exceeds_acceptable | available: 24708 | positive: 6830
Calcium (mg/L)__exceeds_permissible | available: 24708 | positive: 852
Magnesium (mg/L)__exceeds_acceptable | available: 24708 | positive: 11931
Magnesium (mg/L)__exceeds_permissible | available: 24708 | positive: 2167
Sulphate (mg/L)__exceeds_acceptable | available: 24708 | positive: 2730
Sulphate (mg/L)__exceeds_permissible | available: 24708 | positive: 2525


C:\Users\pokep\AppData\Local\Temp\ipykernel_17944\536782683.py:66: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  screening_test_df.drop(columns="_merge", inplace=True)


In [14]:
# CELL 14 — Validate Phase 5 labels against numeric thresholds

def get_threshold(parameter, column):
    row = threshold_registry.loc[
        threshold_registry["Parameter"].eq(parameter)
    ]

    if len(row) != 1:
        raise ValueError(
            f"Expected one threshold row for {parameter}"
        )

    value = pd.to_numeric(
        row.iloc[0][column], errors="coerce"
    )

    if pd.isna(value):
        return None

    return float(value)


label_parity_records = []

for parameter, spec in SCREENING_TARGETS.items():

    actual = pd.to_numeric(
        screening_test_df[f"{spec['prediction_prefix']} - Actual"],
        errors="coerce"
    )

    threshold_row = threshold_registry.loc[
        threshold_registry["Parameter"].eq(parameter)
    ].iloc[0]

    for suffix in spec["labels"]:

        label_col = f"{parameter}__{suffix}"
        saved_label = pd.to_numeric(
            screening_test_df[label_col], errors="coerce"
        )

        if parameter == "Potential of Hydrogen (pH)":
            lower = get_threshold(
                parameter, "Acceptable_Lower"
            )
            upper = get_threshold(
                parameter, "Acceptable_Upper"
            )
            direct_flag = (actual < lower) | (actual > upper)

        else:
            threshold_column = (
                "Acceptable_Max"
                if suffix == "exceeds_acceptable"
                else "Permissible_Max"
            )

            threshold = get_threshold(
                parameter, threshold_column
            )

            if threshold is None:
                raise ValueError(
                    f"Missing threshold for {parameter}: {suffix}"
                )

            direct_flag = actual > threshold

        eligible = saved_label.notna() & actual.notna()

        saved_positive = saved_label.loc[eligible].eq(1)
        direct_positive = direct_flag.loc[eligible]

        mismatches = saved_positive.ne(direct_positive)

        label_parity_records.append({
            "Parameter": parameter,
            "Label": suffix,
            "Eligible_records": int(eligible.sum()),
            "Saved_positive": int(saved_positive.sum()),
            "Direct_threshold_positive": int(direct_positive.sum()),
            "Agreement_percentage": (
                100 * (1 - mismatches.mean())
                if eligible.sum() else np.nan
            ),
            "Mismatch_count": int(mismatches.sum())
        })

label_parity = pd.DataFrame(label_parity_records)

print("PHASE 5 LABEL CONSISTENCY CHECK")
display(label_parity.round(4))

print("\nMISMATCH DETAILS — IF ANY")

for _, row in label_parity.loc[
    label_parity["Mismatch_count"] > 0
].iterrows():

    parameter = row["Parameter"]
    suffix = row["Label"]
    label_col = f"{parameter}__{suffix}"

    print(f"\n{parameter} — {suffix}")

    actual_col = f"{parameter} - Actual"
    threshold_col = (
        "Acceptable_Max"
        if suffix == "exceeds_acceptable"
        else "Permissible_Max"
    )

    threshold = get_threshold(parameter, threshold_col)

    if parameter == "Potential of Hydrogen (pH)":
        lower = get_threshold(parameter, "Acceptable_Lower")
        upper = get_threshold(parameter, "Acceptable_Upper")
        direct_flag = (
            pd.to_numeric(screening_test_df[actual_col], errors="coerce")
            .lt(lower)
            | pd.to_numeric(
                screening_test_df[actual_col], errors="coerce"
            ).gt(upper)
        )
    else:
        direct_flag = (
            pd.to_numeric(
                screening_test_df[actual_col], errors="coerce"
            ) > threshold
        )

    saved_label = pd.to_numeric(
        screening_test_df[label_col], errors="coerce"
    )

    mismatch_mask = (
        saved_label.notna()
        & direct_flag.notna()
        & saved_label.eq(1).ne(direct_flag)
    )

    display(
        screening_test_df.loc[
            mismatch_mask,
            [
                "_record_id",
                actual_col,
                label_col
            ]
        ].head(10)
    )

PHASE 5 LABEL CONSISTENCY CHECK


,Parameter,Label,Eligible_records,Saved_positive,Direct_threshold_positive,Agreement_percentage,Mismatch_count
0,Potential of Hydrogen (pH),exceeds_acceptable,24708,1292,1292,100.0000,0
1,Chloride (mg/L),exceeds_acceptable,24708,4758,4758,100.0000,0
2,Chloride (mg/L),exceeds_permissible,24708,867,867,100.0000,0
3,Calcium (mg/L),exceeds_acceptable,24708,6830,6830,100.0000,0
4,Calcium (mg/L),exceeds_permissible,24708,852,852,100.0000,0
5,Magnesium (mg/L),exceeds_acceptable,24708,11931,11931,100.0000,0
6,Magnesium (mg/L),exceeds_permissible,24708,2167,2167,100.0000,0
7,Sulphate (mg/L),exceeds_acceptable,24708,2730,2730,100.0000,0
8,Sulphate (mg/L),exceeds_permissible,24708,2525,1175,94.5362,1350



MISMATCH DETAILS — IF ANY

Sulphate (mg/L) — exceeds_permissible


,_record_id,Sulphate (mg/L) - Actual,Sulphate (mg/L)__exceeds_permissible
139,gwq_chemical_parameter_manual_cgwb_ap_1961_202...,264.0,1.0
155,gwq_chemical_parameter_manual_cgwb_ap_1961_202...,288.0,1.0
157,gwq_chemical_parameter_manual_cgwb_ap_1961_202...,212.0,1.0
160,gwq_chemical_parameter_manual_cgwb_ap_1961_202...,264.0,1.0
184,gwq_chemical_parameter_manual_cgwb_ap_1961_202...,353.0,1.0
210,gwq_chemical_parameter_manual_cgwb_ap_1961_202...,275.0,1.0
218,gwq_chemical_parameter_manual_cgwb_ap_1961_202...,341.0,1.0
222,gwq_chemical_parameter_manual_cgwb_ap_1961_202...,346.0,1.0
241,gwq_chemical_parameter_manual_cgwb_ap_1961_202...,310.0,1.0
243,gwq_chemical_parameter_manual_cgwb_ap_1961_202...,350.0,1.0


In [16]:
# CELL 15 — Threshold-based false-reassurance evaluation

screening_results = []

for parameter, spec in SCREENING_TARGETS.items():

    actual = pd.to_numeric(
        screening_test_df[f"{spec['prediction_prefix']} - Actual"],
        errors="coerce"
    )

    predicted = pd.to_numeric(
        screening_test_df[
            f"{spec['prediction_prefix']} - Assisted prediction"
        ],
        errors="coerce"
    )

    threshold_row = threshold_registry.loc[
        threshold_registry["Parameter"].eq(parameter)
    ].iloc[0]

    for suffix in spec["labels"]:

        label_col = f"{parameter}__{suffix}"

        actual_label = pd.to_numeric(
            screening_test_df[label_col],
            errors="coerce"
        )

        if parameter == "Potential of Hydrogen (pH)":
            lower = get_threshold(
                parameter, "Acceptable_Lower"
            )
            upper = get_threshold(
                parameter, "Acceptable_Upper"
            )

            predicted_flag = (
                (predicted < lower) | (predicted > upper)
            )

            threshold_description = (
                f"Outside {lower}–{upper} pH"
            )

        else:
            threshold_column = (
                "Acceptable_Max"
                if suffix == "exceeds_acceptable"
                else "Permissible_Max"
            )

            threshold = get_threshold(
                parameter, threshold_column
            )

            predicted_flag = predicted > threshold

            threshold_description = f"> {threshold:g}"

        # Use Phase 5's label-availability mask. This preserves
        # conditional evaluability, especially for sulphate.
        eligible = (
            actual_label.notna()
            & actual.notna()
            & predicted.notna()
        )

        y_true = actual_label.loc[eligible].eq(1).to_numpy()
        y_pred = predicted_flag.loc[eligible].to_numpy(dtype=bool)

        tp = int((y_true & y_pred).sum())
        tn = int((~y_true & ~y_pred).sum())
        fp = int((~y_true & y_pred).sum())
        fn = int((y_true & ~y_pred).sum())

        positive_count = tp + fn
        negative_count = tn + fp

        recall = tp / positive_count if positive_count else np.nan
        specificity = (
            tn / negative_count if negative_count else np.nan
        )
        precision = (
            tp / (tp + fp) if (tp + fp) else np.nan
        )
        false_reassurance_rate = (
            fn / positive_count if positive_count else np.nan
        )

        screening_results.append({
            "Parameter": parameter,
            "Screening_rule": suffix,
            "Threshold": threshold_description,
            "Evaluated_records": int(eligible.sum()),
            "Actual_exceedances": positive_count,
            "Predicted_exceedances": int(y_pred.sum()),
            "True_positives": tp,
            "False_positives": fp,
            "False_negatives": fn,
            "True_negatives": tn,
            "Detection_recall_percentage": (
                100 * recall if pd.notna(recall) else np.nan
            ),
            "False_reassurance_percentage": (
                100 * false_reassurance_rate
                if pd.notna(false_reassurance_rate) else np.nan
            ),
            "Specificity_percentage": (
                100 * specificity
                if pd.notna(specificity) else np.nan
            ),
            "Precision_percentage": (
                100 * precision
                if pd.notna(precision) else np.nan
            )
        })

screening_evaluation = pd.DataFrame(screening_results)

print("ASSISTED MODEL — SCREENING EXCEEDANCE EVALUATION")
display(
    screening_evaluation
    .sort_values(
        ["Parameter", "Screening_rule"]
    )
    .round(3)
)

print("\nIMPORTANT")
print(
    "A false negative means an actual Phase 5 screening label is positive "
    "while the assisted model prediction is below the corresponding "
    "screening threshold."
)
print(
    "This is a retrospective threshold comparison, not a validated "
    "clinical or regulatory safety classification."
)

ASSISTED MODEL — SCREENING EXCEEDANCE EVALUATION


,Parameter,Screening_rule,Threshold,Evaluated_records,Actual_exceedances,Predicted_exceedances,True_positives,False_positives,False_negatives,True_negatives,Detection_recall_percentage,False_reassurance_percentage,Specificity_percentage,Precision_percentage
3,Calcium (mg/L),exceeds_acceptable,> 75,24708,6830,6029,5282,747,1548,17131,77.335,22.665,95.822,87.610
4,Calcium (mg/L),exceeds_permissible,> 200,24708,852,500,404,96,448,23760,47.418,52.582,99.598,80.800
1,Chloride (mg/L),exceeds_acceptable,> 250,24708,4758,4614,4280,334,478,19616,89.954,10.046,98.326,92.761
2,Chloride (mg/L),exceeds_permissible,> 1000,24708,867,797,757,40,110,23801,87.313,12.687,99.832,94.981
5,Magnesium (mg/L),exceeds_acceptable,> 30,24708,11931,11747,10677,1070,1254,11707,89.490,10.510,91.626,90.891
6,Magnesium (mg/L),exceeds_permissible,> 100,24708,2167,1744,1519,225,648,22316,70.097,29.903,99.002,87.099
0,Potential of Hydrogen (pH),exceeds_acceptable,Outside 6.5–8.5 pH,24708,1292,365,282,83,1010,23333,21.827,78.173,99.646,77.260
7,Sulphate (mg/L),exceeds_acceptable,> 200,24708,2730,2116,1846,270,884,21708,67.619,32.381,98.771,87.240
8,Sulphate (mg/L),exceeds_permissible,> 400,24708,2525,829,803,26,1722,22157,31.802,68.198,99.883,96.864



IMPORTANT
A false negative means an actual Phase 5 screening label is positive while the assisted model prediction is below the corresponding screening threshold.
This is a retrospective threshold comparison, not a validated clinical or regulatory safety classification.


In [17]:
# CELL 16 — Audit sulphate permissible-label semantics

sulphate_actual = pd.to_numeric(
    screening_test_df["Sulphate (mg/L) - Actual"],
    errors="coerce"
)

magnesium_actual = pd.to_numeric(
    screening_test_df["Magnesium (mg/L) - Actual"],
    errors="coerce"
)

saved_sulphate_flag = pd.to_numeric(
    screening_test_df[
        "Sulphate (mg/L)__exceeds_permissible"
    ],
    errors="coerce"
)

eligible = (
    sulphate_actual.notna()
    & magnesium_actual.notna()
    & saved_sulphate_flag.notna()
)

s = sulphate_actual.loc[eligible]
m = magnesium_actual.loc[eligible]
y = saved_sulphate_flag.loc[eligible].eq(1)

# Candidate 1: permissible limit is 400 when Mg <= 30,
# and 200 when Mg > 30.
candidate_1 = np.where(m <= 30, s > 400, s > 200)

# Candidate 2: reverse conditional interpretation.
candidate_2 = np.where(m <= 30, s > 200, s > 400)

# Candidate 3: 400 generally, with an additional flag
# when both Mg > 30 and sulphate > 200.
candidate_3 = (s > 400) | ((m > 30) & (s > 200))

# Candidate 4: unconditional sulphate > 400.
candidate_4 = s > 400

candidates = {
    "Mg<=30:400; Mg>30:200": candidate_1,
    "Mg<=30:200; Mg>30:400": candidate_2,
    "400 OR (Mg>30 AND sulphate>200)": candidate_3,
    "Unconditional sulphate>400": candidate_4
}

candidate_records = []

for name, candidate in candidates.items():
    candidate = np.asarray(candidate, dtype=bool)
    mismatch_count = int(np.sum(candidate != y.to_numpy()))

    candidate_records.append({
        "Candidate_rule": name,
        "Eligible_records": len(y),
        "Saved_positive": int(y.sum()),
        "Candidate_positive": int(candidate.sum()),
        "Mismatch_count": mismatch_count,
        "Agreement_percentage": (
            100 * (1 - mismatch_count / len(y))
        )
    })

candidate_comparison = pd.DataFrame(candidate_records)

print("CONDITIONAL RULE CANDIDATE COMPARISON")
display(
    candidate_comparison.sort_values(
        "Mismatch_count"
    ).round(4)
)

# Examine positive labels below the simple 400 mg/L threshold.
mismatch_mask = (
    eligible
    & saved_sulphate_flag.eq(1)
    & sulphate_actual.le(400)
)

mismatch_examples = screening_test_df.loc[
    mismatch_mask,
    [
        "_record_id",
        "Sulphate (mg/L) - Actual",
        "Magnesium (mg/L) - Actual",
        "Sulphate (mg/L)__exceeds_acceptable",
        "Sulphate (mg/L)__exceeds_permissible"
    ]
].copy()

print("\nSAVED PERMISSIBLE POSITIVES AT SULPHATE <= 400")
print("Count:", len(mismatch_examples))

display(
    mismatch_examples
    .sort_values(
        ["Magnesium (mg/L) - Actual",
         "Sulphate (mg/L) - Actual"],
        ascending=False
    )
    .head(20)
)

# Show positive-label frequency across Mg bands.
audit_df = pd.DataFrame({
    "Magnesium": m,
    "Sulphate": s,
    "Saved_flag": y.astype(int)
})

audit_df["Magnesium_band"] = pd.cut(
    audit_df["Magnesium"],
    bins=[-np.inf, 30, np.inf],
    labels=["Mg <= 30", "Mg > 30"]
)

print("\nSULPHATE LABELS BY MAGNESIUM CONDITION")
display(
    audit_df.groupby(
        "Magnesium_band",
        observed=False
    ).agg(
        Records=("Saved_flag", "size"),
        Mean_magnesium=("Magnesium", "mean"),
        Mean_sulphate=("Sulphate", "mean"),
        Saved_positive=("Saved_flag", "sum"),
        Positive_percentage=("Saved_flag", "mean")
    ).assign(
        Positive_percentage=lambda x:
        x["Positive_percentage"] * 100
    ).round(3)
)

CONDITIONAL RULE CANDIDATE COMPARISON


,Candidate_rule,Eligible_records,Saved_positive,Candidate_positive,Mismatch_count,Agreement_percentage
0,Mg<=30:400; Mg>30:200,24708,2525,2525,0,100.0000
2,400 OR (Mg>30 AND sulphate>200),24708,2525,2525,0,100.0000
3,Unconditional sulphate>400,24708,2525,1175,1350,94.5362
1,Mg<=30:200; Mg>30:400,24708,2525,1380,1555,93.7065



SAVED PERMISSIBLE POSITIVES AT SULPHATE <= 400
Count: 1350


,_record_id,Sulphate (mg/L) - Actual,Magnesium (mg/L) - Actual,Sulphate (mg/L)__exceeds_acceptable,Sulphate (mg/L)__exceeds_permissible
5214,gwq_chemical_parameter_manual_cgwb_gj_1961_202...,318.0,1520.00,1.0,1.0
20667,gwq_chemical_parameter_manual_cgwb_tn_1961_202...,265.0,1347.41,1.0,1.0
20672,gwq_chemical_parameter_manual_cgwb_tn_1961_202...,254.0,691.00,1.0,1.0
7451,gwq_chemical_parameter_manual_cgwb_ka_1961_202...,268.0,653.00,1.0,1.0
600,gwq_chemical_parameter_manual_cgwb_ap_1961_202...,400.0,552.00,1.0,1.0
20665,gwq_chemical_parameter_manual_cgwb_tn_1961_202...,317.0,462.73,1.0,1.0
3654,gwq_chemical_parameter_manual_cgwb_dl_1961_202...,350.0,444.00,1.0,1.0
6081,gwq_chemical_parameter_manual_cgwb_hr_1961_202...,389.0,443.00,1.0,1.0
22511,gwq_chemical_parameter_manual_cgwb_up_1961_202...,236.0,428.30,1.0,1.0
6164,gwq_chemical_parameter_manual_cgwb_hr_1961_202...,251.0,428.00,1.0,1.0



SULPHATE LABELS BY MAGNESIUM CONDITION


,Records,Mean_magnesium,Mean_sulphate,Saved_positive,Positive_percentage
Magnesium_band,,,,,
Mg <= 30,12777,15.296,34.484,43,0.337
Mg > 30,11931,76.344,165.733,2482,20.803


In [18]:
# CELL 17 — Corrected threshold-based false-reassurance evaluation

corrected_screening_results = []

for parameter, spec in SCREENING_TARGETS.items():

    actual = pd.to_numeric(
        screening_test_df[f"{spec['prediction_prefix']} - Actual"],
        errors="coerce"
    )

    predicted = pd.to_numeric(
        screening_test_df[
            f"{spec['prediction_prefix']} - Assisted prediction"
        ],
        errors="coerce"
    )

    for suffix in spec["labels"]:

        label_col = f"{parameter}__{suffix}"

        actual_label = pd.to_numeric(
            screening_test_df[label_col],
            errors="coerce"
        )

        # Apply the corresponding rule to model predictions.
        if parameter == "Potential of Hydrogen (pH)":

            lower = get_threshold(
                parameter, "Acceptable_Lower"
            )
            upper = get_threshold(
                parameter, "Acceptable_Upper"
            )

            actual_flag = (
                (actual < lower) | (actual > upper)
            )

            predicted_flag = (
                (predicted < lower) | (predicted > upper)
            )

            threshold_description = f"Outside {lower}–{upper}"
            conditioning_input = "None"

        elif parameter == "Sulphate (mg/L)":

            actual_mg = pd.to_numeric(
                screening_test_df["Magnesium (mg/L) - Actual"],
                errors="coerce"
            )

            permissible = suffix == "exceeds_permissible"

            if permissible:
                # Phase 5 condition:
                # Mg <= 30: sulphate > 400
                # Mg > 30: sulphate > 200
                actual_flag = np.where(
                    actual_mg <= 30,
                    actual > 400,
                    actual > 200
                )

                predicted_flag = np.where(
                    actual_mg <= 30,
                    predicted > 400,
                    predicted > 200
                )

                threshold_description = (
                    "Mg <= 30: >400; Mg >30: >200"
                )
                conditioning_input = "Actual measured magnesium"

            else:
                actual_flag = actual > 200
                predicted_flag = predicted > 200

                threshold_description = ">200"
                conditioning_input = "None"

        else:
            threshold_column = (
                "Acceptable_Max"
                if suffix == "exceeds_acceptable"
                else "Permissible_Max"
            )

            threshold = get_threshold(
                parameter, threshold_column
            )

            actual_flag = actual > threshold
            predicted_flag = predicted > threshold

            threshold_description = f">{threshold:g}"
            conditioning_input = "None"

        # Independently verify the actual measurements against
        # the stored Phase 5 labels.
        parity_mask = (
            actual_label.notna()
            & actual.notna()
        )

        parity_mismatches = int(
            (
                actual_label.loc[parity_mask].eq(1).to_numpy()
                != np.asarray(actual_flag)[parity_mask.to_numpy()]
            ).sum()
        )

        if parity_mismatches != 0:
            raise AssertionError(
                f"Threshold-label mismatch for {parameter} / {suffix}: "
                f"{parity_mismatches}"
            )

        eligible = (
            actual_label.notna()
            & actual.notna()
            & predicted.notna()
        )

        if parameter == "Sulphate (mg/L)" and suffix == "exceeds_permissible":
            eligible &= actual_mg.notna()

        y_true = actual_label.loc[eligible].eq(1).to_numpy()

        y_pred = np.asarray(predicted_flag)[
            eligible.to_numpy()
        ].astype(bool)

        tp = int((y_true & y_pred).sum())
        tn = int((~y_true & ~y_pred).sum())
        fp = int((~y_true & y_pred).sum())
        fn = int((y_true & ~y_pred).sum())

        positive_count = tp + fn
        negative_count = tn + fp

        recall = (
            tp / positive_count
            if positive_count else np.nan
        )

        specificity = (
            tn / negative_count
            if negative_count else np.nan
        )

        precision = (
            tp / (tp + fp)
            if (tp + fp) else np.nan
        )

        false_reassurance = (
            fn / positive_count
            if positive_count else np.nan
        )

        corrected_screening_results.append({
            "Parameter": parameter,
            "Screening_rule": suffix,
            "Threshold": threshold_description,
            "Conditioning_input": conditioning_input,
            "Label_parity_mismatches": parity_mismatches,
            "Evaluated_records": int(eligible.sum()),
            "Actual_exceedances": positive_count,
            "Predicted_exceedances": int(y_pred.sum()),
            "True_positives": tp,
            "False_positives": fp,
            "False_negatives": fn,
            "True_negatives": tn,
            "Detection_recall_percentage": (
                100 * recall if pd.notna(recall) else np.nan
            ),
            "False_reassurance_percentage": (
                100 * false_reassurance
                if pd.notna(false_reassurance) else np.nan
            ),
            "Specificity_percentage": (
                100 * specificity
                if pd.notna(specificity) else np.nan
            ),
            "Precision_percentage": (
                100 * precision
                if pd.notna(precision) else np.nan
            )
        })

corrected_screening_evaluation = pd.DataFrame(
    corrected_screening_results
)

print("CORRECTED SCREENING EVALUATION")
display(
    corrected_screening_evaluation
    .sort_values(["Parameter", "Screening_rule"])
    .round(3)
)

assert (
    corrected_screening_evaluation[
        "Label_parity_mismatches"
    ].eq(0).all()
)

print("\nVALIDATION PASSED")
print("All evaluated threshold rules reproduce Phase 5 actual labels.")
print(
    "Sulphate permissible evaluation uses actual measured magnesium "
    "as the conditioning input."
)

CORRECTED SCREENING EVALUATION


,Parameter,Screening_rule,Threshold,Conditioning_input,Label_parity_mismatches,Evaluated_records,Actual_exceedances,Predicted_exceedances,True_positives,False_positives,False_negatives,True_negatives,Detection_recall_percentage,False_reassurance_percentage,Specificity_percentage,Precision_percentage
3,Calcium (mg/L),exceeds_acceptable,>75,None,0,24708,6830,6029,5282,747,1548,17131,77.335,22.665,95.822,87.610
4,Calcium (mg/L),exceeds_permissible,>200,None,0,24708,852,500,404,96,448,23760,47.418,52.582,99.598,80.800
1,Chloride (mg/L),exceeds_acceptable,>250,None,0,24708,4758,4614,4280,334,478,19616,89.954,10.046,98.326,92.761
2,Chloride (mg/L),exceeds_permissible,>1000,None,0,24708,867,797,757,40,110,23801,87.313,12.687,99.832,94.981
5,Magnesium (mg/L),exceeds_acceptable,>30,None,0,24708,11931,11747,10677,1070,1254,11707,89.490,10.510,91.626,90.891
6,Magnesium (mg/L),exceeds_permissible,>100,None,0,24708,2167,1744,1519,225,648,22316,70.097,29.903,99.002,87.099
0,Potential of Hydrogen (pH),exceeds_acceptable,Outside 6.5–8.5,None,0,24708,1292,365,282,83,1010,23333,21.827,78.173,99.646,77.260
7,Sulphate (mg/L),exceeds_acceptable,>200,None,0,24708,2730,2116,1846,270,884,21708,67.619,32.381,98.771,87.240
8,Sulphate (mg/L),exceeds_permissible,Mg <= 30: >400; Mg >30: >200,Actual measured magnesium,0,24708,2525,2011,1759,252,766,21931,69.663,30.337,98.864,87.469



VALIDATION PASSED
All evaluated threshold rules reproduce Phase 5 actual labels.
Sulphate permissible evaluation uses actual measured magnesium as the conditioning input.


In [19]:
# CELL 18 — Compare screening detection across both model types

model_screening_records = []

for parameter, spec in SCREENING_TARGETS.items():

    actual = pd.to_numeric(
        screening_test_df[f"{spec['prediction_prefix']} - Actual"],
        errors="coerce"
    )

    for model in ["Context", "Assisted"]:

        predicted = pd.to_numeric(
            screening_test_df[
                f"{spec['prediction_prefix']} - {model} prediction"
            ],
            errors="coerce"
        )

        for suffix in spec["labels"]:

            label_col = f"{parameter}__{suffix}"

            actual_label = pd.to_numeric(
                screening_test_df[label_col],
                errors="coerce"
            )

            if parameter == "Potential of Hydrogen (pH)":
                lower = get_threshold(parameter, "Acceptable_Lower")
                upper = get_threshold(parameter, "Acceptable_Upper")

                predicted_flag = (
                    (predicted < lower) | (predicted > upper)
                )

                threshold_description = f"Outside {lower}–{upper}"

            elif parameter == "Sulphate (mg/L)" and suffix == "exceeds_permissible":

                actual_mg = pd.to_numeric(
                    screening_test_df["Magnesium (mg/L) - Actual"],
                    errors="coerce"
                )

                predicted_flag = np.where(
                    actual_mg <= 30,
                    predicted > 400,
                    predicted > 200
                )

                threshold_description = (
                    "Mg <= 30: >400; Mg >30: >200"
                )

            else:
                threshold_column = (
                    "Acceptable_Max"
                    if suffix == "exceeds_acceptable"
                    else "Permissible_Max"
                )

                threshold = get_threshold(
                    parameter, threshold_column
                )

                predicted_flag = predicted > threshold
                threshold_description = f">{threshold:g}"

            eligible = (
                actual_label.notna()
                & actual.notna()
                & predicted.notna()
            )

            if (
                parameter == "Sulphate (mg/L)"
                and suffix == "exceeds_permissible"
            ):
                eligible &= actual_mg.notna()

            y_true = actual_label.loc[eligible].eq(1).to_numpy()
            y_pred = np.asarray(predicted_flag)[
                eligible.to_numpy()
            ].astype(bool)

            tp = int((y_true & y_pred).sum())
            tn = int((~y_true & ~y_pred).sum())
            fp = int((~y_true & y_pred).sum())
            fn = int((y_true & ~y_pred).sum())

            positive_count = tp + fn
            negative_count = tn + fp

            model_screening_records.append({
                "Parameter": parameter,
                "Screening_rule": suffix,
                "Threshold": threshold_description,
                "Model": model,
                "Evaluated_records": int(eligible.sum()),
                "Actual_exceedances": positive_count,
                "Predicted_exceedances": int(y_pred.sum()),
                "True_positives": tp,
                "False_positives": fp,
                "False_negatives": fn,
                "True_negatives": tn,
                "Recall_percentage": (
                    100 * tp / positive_count
                    if positive_count else np.nan
                ),
                "Miss_rate_percentage": (
                    100 * fn / positive_count
                    if positive_count else np.nan
                ),
                "Specificity_percentage": (
                    100 * tn / negative_count
                    if negative_count else np.nan
                ),
                "Precision_percentage": (
                    100 * tp / (tp + fp)
                    if (tp + fp) else np.nan
                )
            })

model_screening_comparison = pd.DataFrame(
    model_screening_records
)

print("SCREENING PERFORMANCE — BOTH MODELS")
display(
    model_screening_comparison.sort_values(
        ["Parameter", "Screening_rule", "Model"]
    ).round(3)
)

# Side-by-side comparison
screening_recall_comparison = (
    model_screening_comparison
    .pivot(
        index=["Parameter", "Screening_rule"],
        columns="Model",
        values=["Recall_percentage", "Miss_rate_percentage"]
    )
)

screening_recall_comparison.columns = [
    f"{metric}_{model}"
    for metric, model in screening_recall_comparison.columns
]

screening_recall_comparison = (
    screening_recall_comparison.reset_index()
)

screening_recall_comparison["Assisted_recall_change_pp"] = (
    screening_recall_comparison["Recall_percentage_Assisted"]
    - screening_recall_comparison["Recall_percentage_Context"]
)

screening_recall_comparison["Assisted_miss_rate_change_pp"] = (
    screening_recall_comparison["Miss_rate_percentage_Assisted"]
    - screening_recall_comparison["Miss_rate_percentage_Context"]
)

print("\nCONTEXT VS ASSISTED — RECALL COMPARISON")
display(
    screening_recall_comparison.sort_values(
        ["Parameter", "Screening_rule"]
    ).round(3)
)

print("\nSUMMARY")
print(
    "Screening comparisons where assisted recall is higher:",
    int(
        screening_recall_comparison[
            "Assisted_recall_change_pp"
        ].gt(0).sum()
    ),
    "out of",
    len(screening_recall_comparison)
)

print(
    "Screening comparisons where assisted miss rate is lower:",
    int(
        screening_recall_comparison[
            "Assisted_miss_rate_change_pp"
        ].lt(0).sum()
    ),
    "out of",
    len(screening_recall_comparison)
)

SCREENING PERFORMANCE — BOTH MODELS


,Parameter,Screening_rule,Threshold,Model,Evaluated_records,Actual_exceedances,Predicted_exceedances,True_positives,False_positives,False_negatives,True_negatives,Recall_percentage,Miss_rate_percentage,Specificity_percentage,Precision_percentage
8,Calcium (mg/L),exceeds_acceptable,>75,Assisted,24708,6830,6029,5282,747,1548,17131,77.335,22.665,95.822,87.610
6,Calcium (mg/L),exceeds_acceptable,>75,Context,24708,6830,4207,2415,1792,4415,16086,35.359,64.641,89.977,57.404
9,Calcium (mg/L),exceeds_permissible,>200,Assisted,24708,852,500,404,96,448,23760,47.418,52.582,99.598,80.800
7,Calcium (mg/L),exceeds_permissible,>200,Context,24708,852,50,21,29,831,23827,2.465,97.535,99.878,42.000
4,Chloride (mg/L),exceeds_acceptable,>250,Assisted,24708,4758,4614,4280,334,478,19616,89.954,10.046,98.326,92.761
2,Chloride (mg/L),exceeds_acceptable,>250,Context,24708,4758,3034,1859,1175,2899,18775,39.071,60.929,94.110,61.272
5,Chloride (mg/L),exceeds_permissible,>1000,Assisted,24708,867,797,757,40,110,23801,87.313,12.687,99.832,94.981
3,Chloride (mg/L),exceeds_permissible,>1000,Context,24708,867,154,76,78,791,23763,8.766,91.234,99.673,49.351
12,Magnesium (mg/L),exceeds_acceptable,>30,Assisted,24708,11931,11747,10677,1070,1254,11707,89.490,10.510,91.626,90.891
10,Magnesium (mg/L),exceeds_acceptable,>30,Context,24708,11931,12282,8749,3533,3182,9244,73.330,26.670,72.349,71.234



CONTEXT VS ASSISTED — RECALL COMPARISON


,Parameter,Screening_rule,Recall_percentage_Assisted,Recall_percentage_Context,Miss_rate_percentage_Assisted,Miss_rate_percentage_Context,Assisted_recall_change_pp,Assisted_miss_rate_change_pp
0,Calcium (mg/L),exceeds_acceptable,77.335,35.359,22.665,64.641,41.977,-41.977
1,Calcium (mg/L),exceeds_permissible,47.418,2.465,52.582,97.535,44.953,-44.953
2,Chloride (mg/L),exceeds_acceptable,89.954,39.071,10.046,60.929,50.883,-50.883
3,Chloride (mg/L),exceeds_permissible,87.313,8.766,12.687,91.234,78.547,-78.547
4,Magnesium (mg/L),exceeds_acceptable,89.490,73.330,10.510,26.670,16.160,-16.160
5,Magnesium (mg/L),exceeds_permissible,70.097,9.183,29.903,90.817,60.914,-60.914
6,Potential of Hydrogen (pH),exceeds_acceptable,21.827,22.755,78.173,77.245,-0.929,0.929
7,Sulphate (mg/L),exceeds_acceptable,67.619,24.359,32.381,75.641,43.260,-43.260
8,Sulphate (mg/L),exceeds_permissible,69.663,24.515,30.337,75.485,45.149,-45.149



SUMMARY
Screening comparisons where assisted recall is higher: 8 out of 9
Screening comparisons where assisted miss rate is lower: 8 out of 9


In [20]:
# CELL 19 — Consolidated model validation findings

performance_wide = recalculated_metrics.pivot(
    index="Target",
    columns="Model",
    values=["MAE", "RMSE", "R2", "Mean_signed_error"]
)

performance_wide.columns = [
    f"{metric}_{model}"
    for metric, model in performance_wide.columns
]

performance_wide = performance_wide.reset_index()

screening_summary = (
    corrected_screening_evaluation
    .groupby("Parameter")
    .agg(
        Screening_rules=("Screening_rule", "count"),
        Worst_false_reassurance_percentage=(
            "False_reassurance_percentage", "max"
        ),
        Best_detection_recall_percentage=(
            "Detection_recall_percentage", "max"
        )
    )
    .reset_index()
    .rename(columns={"Parameter": "Target"})
)

phase9_target_summary = performance_wide.merge(
    screening_summary,
    on="Target",
    how="left",
    validate="one_to_one"
)

recommendations = {
    "Potential of Hydrogen (pH)": (
        "Not suitable for standalone screening. "
        "Assisted recall is only 21.83%."
    ),
    "Electric Conductivity (μS/cm)": (
        "Strong chemistry-assisted regression results. "
        "No directly matching threshold exists in the Phase 5 registry."
    ),
    "Chloride (mg/L)": (
        "Useful supplementary estimate, but laboratory confirmation "
        "remains necessary because some threshold exceedances are missed."
    ),
    "Bicarbonate (mg/L)": (
        "Moderate chemistry-assisted regression performance. "
        "No directly matching threshold exists in the Phase 5 registry."
    ),
    "Sulphate (mg/L)": (
        "Supplementary estimate only. High-value errors and conditional "
        "threshold misses require particular caution."
    ),
    "Calcium (mg/L)": (
        "Not suitable for standalone permissible-limit screening. "
        "More than half of actual permissible exceedances are missed."
    ),
    "Magnesium (mg/L)": (
        "Useful supplementary estimate, but permissible-limit misses "
        "remain material."
    ),
    "Sodium (mg/L)": (
        "Strong chemistry-assisted regression performance. "
        "No directly matching threshold exists in the Phase 5 registry."
    )
}

phase9_target_summary["Validation_recommendation"] = (
    phase9_target_summary["Target"].map(recommendations)
)

print("CONSOLIDATED PHASE 9 TARGET SUMMARY")
display(
    phase9_target_summary.round(4)
)

CONSOLIDATED PHASE 9 TARGET SUMMARY


,Target,MAE_Assisted,MAE_Context,RMSE_Assisted,RMSE_Context,R2_Assisted,R2_Context,Mean_signed_error_Assisted,Mean_signed_error_Context,Screening_rules,Worst_false_reassurance_percentage,Best_detection_recall_percentage,Validation_recommendation
0,Bicarbonate (mg/L),51.0159,100.7022,88.9855,148.8788,0.7727,0.3639,-14.8372,-26.9192,NaN,NaN,NaN,Moderate chemistry-assisted regression perform...
1,Calcium (mg/L),15.1761,31.7489,33.0536,59.8059,0.7411,0.1523,-5.0667,-11.9112,2.0,52.5822,77.3353,Not suitable for standalone permissible-limit ...
2,Chloride (mg/L),32.5106,143.9307,129.8359,435.9246,0.9245,0.1492,-9.3386,-73.8502,2.0,12.6874,89.9538,"Useful supplementary estimate, but laboratory ..."
3,Electric Conductivity (μS/cm),99.2824,646.9939,414.0321,1499.4387,0.9413,0.2304,-22.4300,-238.4872,NaN,NaN,NaN,Strong chemistry-assisted regression results. ...
4,Magnesium (mg/L),10.5780,24.6047,34.0365,57.0753,0.7096,0.1835,-4.0186,-10.7951,2.0,29.9031,89.4896,"Useful supplementary estimate, but permissible..."
5,Potential of Hydrogen (pH),0.2216,0.2255,0.3038,0.3227,0.5418,0.4828,0.0001,0.0041,1.0,78.1734,21.8266,Not suitable for standalone screening. Assiste...
6,Sodium (mg/L),22.0624,90.9004,76.1191,250.1224,0.9285,0.2275,-7.9226,-42.6675,NaN,NaN,NaN,Strong chemistry-assisted regression performan...
7,Sulphate (mg/L),39.8250,70.8188,167.4453,239.5333,0.5702,0.1204,-21.7895,-42.2144,2.0,32.3810,69.6634,Supplementary estimate only. High-value errors...


In [21]:
# CELL 20 — Save Phase 9 validation outputs

PHASE9_DIR = (
    PROJECT_ROOT
    / "reports"
    / "model_evaluation"
    / "phase_9"
)

PHASE9_DIR.mkdir(parents=True, exist_ok=True)

phase9_exports = {
    "recalculated_metrics":
        (metric_recheck, "phase9_recalculated_metrics.csv"),

    "target_summary":
        (phase9_target_summary, "phase9_target_model_summary.csv"),

    "state_metrics":
        (state_metrics, "phase9_state_level_metrics.csv"),

    "regional_summary":
        (regional_summary, "phase9_regional_performance_summary.csv"),

    "state_win_summary":
        (state_win_summary, "phase9_state_model_win_summary.csv"),

    "state_comparison":
        (state_comparison, "phase9_state_head_to_head.csv"),

    "state_coverage":
        (state_coverage_audit, "phase9_state_coverage_audit.csv"),

    "decile_diagnostics":
        (decile_diagnostics, "phase9_decile_residual_diagnostics.csv"),

    "tail_comparison":
        (tail_comparison, "phase9_low_high_value_comparison.csv"),

    "label_parity":
        (label_parity, "phase9_threshold_label_parity.csv"),

    "corrected_screening":
        (
            corrected_screening_evaluation,
            "phase9_corrected_screening_evaluation.csv"
        ),

    "model_screening_comparison":
        (
            model_screening_comparison,
            "phase9_context_vs_assisted_screening.csv"
        ),

    "screening_recall_comparison":
        (
            screening_recall_comparison,
            "phase9_screening_recall_comparison.csv"
        ),

    "phase6_error_ranges":
        (
            phase6_data["error_ranges"],
            "phase9_phase6_error_range_reference.csv"
        ),

    "phase6_feature_importance":
        (
            phase6_data["importance"],
            "phase9_feature_importance_reference.csv"
        )
}

saved_phase9_paths = {}

for name, (df, filename) in phase9_exports.items():
    path = PHASE9_DIR / filename
    df.to_csv(path, index=False)
    saved_phase9_paths[name] = path

# Create a concise methodology and limitations report.
phase9_report = """
BOREWELL SENTINEL INDIA
PHASE 9 — MODEL EVALUATION AND VALIDATION

1. EVALUATION DATA
- Saved spatial holdout predictions: 24,708 records.
- Unique spatial groups in holdout: 4,359.
- States represented: 30.
- States meeting the 50-record regional analysis threshold: 24.
- Records in the six excluded small state groups: 85.
- The recalculated national MAE, RMSE and R2 values reproduce
  the Phase 6 comparison export exactly.

2. REGRESSION PERFORMANCE
- Chemistry-assisted models have lower national MAE and higher
  national R2 than context-only models for all eight targets.
- Chemistry-assisted national R2:
  pH 0.5418; EC 0.9413; chloride 0.9245;
  bicarbonate 0.7727; sulphate 0.5702;
  calcium 0.7411; magnesium 0.7096; sodium 0.9285.
- Chemistry-assisted performance varies by target and region.
- pH is notably less consistent across states than most other targets.

3. EXTREME-VALUE LIMITATION
- All eight chemistry-assisted models underpredict on average
  within the highest actual-value decile.
- Highest-decile MAE is approximately 1.76 to 5.20 times
  the overall MAE, depending on target.
- Predictions show regression towards typical values and can
  underestimate extreme measurements.

4. SCREENING EVALUATION
- Five of the eight regression targets have directly matching
  Phase 5 screening rules: pH, chloride, calcium, magnesium,
  and sulphate.
- No directly matching threshold was used for EC, bicarbonate,
  or sodium.
- The actual-label consistency check has zero mismatches for
  every evaluated screening definition.
- Chemistry-assisted threshold detection improves on context-only
  detection for eight of the nine comparisons.
- pH is the exception: assisted recall is 21.827%, compared with
  22.755% for context-only.
- Assisted false-reassurance rates include:
  pH 78.173%;
  calcium permissible 52.582%;
  magnesium permissible 29.903%;
  sulphate conditional permissible 30.337%.

5. SULPHATE CONDITIONAL RULE
The Phase 5 permissible sulphate label is reproduced exactly by:
- Magnesium <= 30 mg/L: sulphate > 400 mg/L
- Magnesium > 30 mg/L: sulphate > 200 mg/L

The corrected test-set comparison uses actual measured magnesium
as the conditioning input. This evaluates the conditional label
semantics and should not be confused with a stand-alone sulphate
prediction that has no magnesium measurement available.

6. PRACTICAL INTERPRETATION
- Chemistry-assisted estimates are supplementary analytical outputs
  when the other seven chemistry measurements are available.
- Context-only predictions are substantially weaker for most targets.
- Neither model should independently clear a sample as compliant.
- Actual laboratory measurements must take precedence.
- High-value underprediction and screening false negatives require
  explicit warnings in any future application.
- EC, bicarbonate and sodium must not inherit thresholds from
  different parameters.

7. IMPORTANT LIMITATIONS
- This is retrospective evaluation on one spatial holdout.
- No independent external dataset has been validated here.
- The Phase 5 threshold registry and label semantics are reproduced,
  not independently certified by this evaluation.
- The sulphate condition remains subject to the documented
  methodological provenance review.
- Performance metrics do not establish field-level water safety.
- Point predictions are not calibrated prediction intervals.
- Results must not be interpreted as causal or health-risk estimates.

PHASE 9 STATUS
Evaluation of the existing saved models and holdout predictions
is complete. Model retraining and improvement are separate future work.
"""

report_path = (
    PHASE9_DIR
    / "phase9_model_evaluation_and_validation_report.txt"
)

report_path.write_text(
    phase9_report.strip(),
    encoding="utf-8"
)

saved_phase9_paths["report"] = report_path

print("PHASE 9 EXPORTS")
for name, path in saved_phase9_paths.items():
    print(f"{name}: {path}")

print("\nOutput directory:", PHASE9_DIR)

PHASE 9 EXPORTS
recalculated_metrics: c:\career\projects\BoreWell Sentinel India\reports\model_evaluation\phase_9\phase9_recalculated_metrics.csv
target_summary: c:\career\projects\BoreWell Sentinel India\reports\model_evaluation\phase_9\phase9_target_model_summary.csv
state_metrics: c:\career\projects\BoreWell Sentinel India\reports\model_evaluation\phase_9\phase9_state_level_metrics.csv
regional_summary: c:\career\projects\BoreWell Sentinel India\reports\model_evaluation\phase_9\phase9_regional_performance_summary.csv
state_win_summary: c:\career\projects\BoreWell Sentinel India\reports\model_evaluation\phase_9\phase9_state_model_win_summary.csv
state_comparison: c:\career\projects\BoreWell Sentinel India\reports\model_evaluation\phase_9\phase9_state_head_to_head.csv
state_coverage: c:\career\projects\BoreWell Sentinel India\reports\model_evaluation\phase_9\phase9_state_coverage_audit.csv
decile_diagnostics: c:\career\projects\BoreWell Sentinel India\reports\model_evaluation\phase_9\

In [22]:
# CELL 21 — Verify Phase 9 outputs and evaluation integrity

assert len(metric_recheck) == 16
assert len(corrected_screening_evaluation) == 9
assert len(model_screening_comparison) == 18
assert len(screening_recall_comparison) == 9

assert metric_recheck[
    ["MAE_difference", "RMSE_difference", "R2_difference"]
].abs().max().max() < 1e-3

assert corrected_screening_evaluation[
    "Label_parity_mismatches"
].eq(0).all()

assert int(state_coverage_audit["Test_records"].sum()) == 24_708
assert int(
    state_coverage_audit.loc[
        state_coverage_audit["Meets_minimum"],
        "Test_records"
    ].sum()
) == 24_623

assert len(saved_phase9_paths) == len(phase9_exports) + 1

missing_phase9_exports = [
    name for name, path in saved_phase9_paths.items()
    if not path.exists() or path.stat().st_size == 0
]

assert not missing_phase9_exports, (
    f"Missing or empty exports: {missing_phase9_exports}"
)

print("PHASE 9 VALIDATION PASSED")
print("- National metrics independently reproduced")
print("- 24,708 test records reconciled")
print("- Regional coverage reconciled")
print("- Nine screening comparisons validated")
print("- Zero threshold-label mismatches")
print("- Context and assisted screening comparison saved")
print("- Decile and extreme-value diagnostics saved")
print("- All Phase 9 export files verified")
print("- Total output files:", len(saved_phase9_paths))
print("- Output directory:", PHASE9_DIR)

PHASE 9 VALIDATION PASSED
- National metrics independently reproduced
- 24,708 test records reconciled
- Regional coverage reconciled
- Nine screening comparisons validated
- Zero threshold-label mismatches
- Context and assisted screening comparison saved
- Decile and extreme-value diagnostics saved
- All Phase 9 export files verified
- Total output files: 16
- Output directory: c:\career\projects\BoreWell Sentinel India\reports\model_evaluation\phase_9


# Phase 9 — Model Evaluation and Validation: Completed

## Objective
Evaluate the predictive performance, regional consistency,
error distribution, and screening limitations of the Phase 6 models.

## Completed evaluation
1. Independent reproduction of MAE, RMSE and R2.
2. Spatial holdout evaluation.
3. State-level performance comparisons.
4. Residual and actual-value decile analysis.
5. Extreme-value underprediction analysis.
6. Threshold-label consistency validation.
7. Corrected sulphate conditional-rule evaluation.
8. Context-only versus chemistry-assisted screening comparison.
9. Consolidated validation report and artifact exports.

## Main findings
- Chemistry-assisted models improve national regression metrics
  across all eight targets.
- Regional performance varies, especially for pH and sulphate.
- High-value measurements are systematically underpredicted.
- Threshold-screening false negatives remain material.
- Chemistry-assisted predictions are supplementary estimates,
  not substitutes for laboratory testing.
- Context-only predictions are not suitable as independent
  groundwater compliance screening.

## Limitations
- No independent external dataset validation has been completed.
- No prediction-interval calibration has been completed.
- Threshold provenance and sulphate conditional interpretation
  require continued documentation.
- The spatial groups and location identities remain provisional.
- Model outputs must never be interpreted as definitive
  water-safety classifications.

## Final status
**PHASE 9 COMPLETED — EXISTING MODEL EVALUATION**

Next:
Phase 10 — Web Application Development.